# INF8225 2026, TP3: RNNs, GRUs and Transformers for Machine translation

***Work author:***

***Reagan - LI ---- 2055734***


***Bruno NAWFAL ---- 2062203***


The goal of this TP is to compare the performance of three different architectures for neural machine translation:
* A vanilla RNN
* A GRU-RNN
* A transformer

You are provided with the code to load and build the pytorch dataset,
and the code for the training loop.
You "only" have to code the GRU and Transformer architectures, the RNN is provided and the notebook should run using the RNN codepath so you can immediately see the model train by running this notebook. There is code to reduce the dataset by some percentage, so the default is to train with only 5% of the data. This leads to fast training but lower performing models.
The use of built-in torch layers such as `nn.GRU`, `nn.RNN` or `nn.Transformer`
is forbidden, as the goal of this TP is to implement the mathematics of the underlying models. You will be given many hints on how to implement the models. Each section that you need to implement is flagged with a TODO statement.

The source sentences in the dataset are in english and the target language is french.

Do not forget to **select the runtime type as GPU!**, and if you want a better GPU you could purchase a small amount of compute to potentially get an H100 or A100 GPU.

**Sources**

* Dataset: [Tab-delimited Bilingual Sentence Pairs](http://www.manythings.org/anki/)

<!---
M. Cettolo, C. Girardi, and M. Federico. 2012. WIT3: Web Inventory of Transcribed and Translated Talks. In Proc. of EAMT, pp. 261-268, Trento, Italy. pdf, bib. [paper](https://aclanthology.org/2012.eamt-1.60.pdf). [website](https://wit3.fbk.eu/2016-01).
-->

* The code is inspired by this [pytorch tutorial](https://pytorch.org/tutorials/beginner/torchtext_translation_tutorial.html).

*This notebook is quite big, use the table of contents to easily navigate through it.*

# Imports


In [1]:
!python3 -m spacy download en_core_web_sm > /dev/null
!python3 -m spacy download fr_core_news_sm > /dev/null
!pip install torchinfo > /dev/null
!pip install einops > /dev/null
!pip install wandb > /dev/null


from itertools import takewhile
from collections import Counter, defaultdict

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

import spacy

import einops
import wandb
from torchinfo import summary

In [2]:
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-30498d47-585c-56f3-3425-accc36734ca8)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Data and tokenization

We first download and parse the dataset. From the parsed sentences
we can build the vocabularies and the torch datasets.
The end goal of this section is to have an iterator
that can yield the pairs of translated datasets, and
where each sentences is made of a sequence of tokens.

The tokenizers are objects that are able to divide a python string into a list of tokens (words, punctuations, special tokens...) as a list of strings.

The special tokens are used for a particular reasons:
* *\<unk\>*: Replace an unknown word in the vocabulary by this default token
* *\<pad\>*: Virtual token used to as padding token so a batch of sentences can have a unique length
* *\<bos\>*: Token indicating the beggining of a sentence in the target sequence
* *\<eos\>*: Token indicating the end of a sentence in the target sequence

In [4]:

# Our current dataset is
!wget http://www.manythings.org/anki/fra-eng.zip
!unzip fra-eng.zip


df = pd.read_csv('fra.txt', sep='\t', names=['english', 'french', 'attribution'])
train = [
    (en, fr) for en, fr in zip(df['english'], df['french'])
]
train, valid = train_test_split(train, test_size=0.1, random_state=0)
print(len(train))

# Optional: Reduce dataset size for faster training
USE_REDUCED_DATASET = True  # Set to True to use smaller dataset
DATASET_FRACTION = 0.33  # Use 5% of data

if USE_REDUCED_DATASET:
    train = train[:int(len(train) * DATASET_FRACTION)]
    print(f"Using reduced training set: {len(train)} examples ({DATASET_FRACTION*100}%)")
else:
    print(f"Using full training set: {len(train)} examples")

print(len(train))

# Load spacy models
en_nlp = spacy.load('en_core_web_sm')
fr_nlp = spacy.load('fr_core_news_sm')

# Create tokenizer functions
def en_tokenizer(text):
    return [token.text for token in en_nlp.tokenizer(text)]

def fr_tokenizer(text):
    return [token.text for token in fr_nlp.tokenizer(text)]

SPECIALS = ['<unk>', '<pad>', '<bos>', '<eos>']


# An older dataset, (there's an issue on Colab with it, so we will not use it)
# train, valid, _ = IWSLT2016(language_pair=('fr', 'en'))
# train, valid = list(train), list(valid)

# Another dataset, but it is too large
# !wget https://www.statmt.org/wmt14/training-monolingual-europarl-v7/europarl-v7.en.gz
# !wget https://www.statmt.org/wmt14/training-monolingual-europarl-v7/europarl-v7.fr.gz
# !gunzip europarl-v7.en.gz
# !gunzip europarl-v7.fr.gz

# with open('europarl-v7.en', 'r') as my_file:
#     english = my_file.readlines()

# with open('europarl-v7.fr', 'r') as my_file:
#     french = my_file.readlines()

# dataset = [
#     (en, fr)
#     for en, fr in zip(english, french)
# ]
# print(f'\n{len(dataset):,} sentences.')

# dataset, _ = train_test_split(dataset, test_size=0.8, random_state=0)  # Remove 80% of the dataset (it would be huge otherwise)
# train, valid = train_test_split(dataset, test_size=0.2, random_state=0)  # Split between train and validation dataset


--2026-04-22 23:09:11--  http://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8242175 (7.9M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   7.86M  4.82MB/s    in 1.6s    

2026-04-22 23:09:13 (4.82 MB/s) - ‘fra-eng.zip’ saved [8242175/8242175]

Archive:  fra-eng.zip
  inflating: _about.txt              
  inflating: fra.txt                 
216468
Using reduced training set: 71434 examples (33.0%)
71434


## Replacement for torchtext tokenization

Functions and classes to build the vocabularies and the torch datasets.
The vocabulary is an object able to transform a string token into the id (an int) of that token in the vocabulary.

In [5]:
# Custom Vocabulary class (replaces torchtext.vocab.Vocab)
class Vocab:
    """Simple vocabulary class for token-to-index mapping."""

    def __init__(self, token_to_idx, default_index=None):
        self.token_to_idx = token_to_idx
        self.idx_to_token = {idx: token for token, idx in token_to_idx.items()}
        self.default_index = default_index

    def __len__(self):
        return len(self.token_to_idx)

    def __getitem__(self, token):
        """Get index of a token."""
        if self.default_index is not None:
            return self.token_to_idx.get(token, self.default_index)
        return self.token_to_idx[token]

    def __call__(self, tokens):
        """Convert list of tokens to list of indices."""
        return [self[token] for token in tokens]

    def set_default_index(self, default_index):
        """Set the default index for unknown tokens."""
        self.default_index = default_index

    def lookup_token(self, idx):
        """Get token from index."""
        return self.idx_to_token.get(idx, '<unk>')

    def lookup_tokens(self, indices):
        """Get list of tokens from list of indices."""
        return [self.lookup_token(idx) for idx in indices]


def build_vocab_from_iterator(iterator, min_freq=1, specials=None):
    """Build vocabulary from an iterator of token lists.

    Args:
        iterator: Iterator yielding lists of tokens
        min_freq: Minimum frequency for a token to be included
        specials: List of special tokens to add at the beginning

    Returns:
        Vocab object
    """
    # Count token frequencies
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)

    # Build token to index mapping
    token_to_idx = {}

    # Add special tokens first
    if specials:
        for token in specials:
            token_to_idx[token] = len(token_to_idx)

    # Add tokens that meet minimum frequency
    for token, freq in counter.items():
        if freq >= min_freq and token not in token_to_idx:
            token_to_idx[token] = len(token_to_idx)

    return Vocab(token_to_idx)

In [6]:
class TranslationDataset(Dataset):
    def __init__(
            self,
            dataset: list,
            en_vocab: Vocab,
            fr_vocab: Vocab,
            en_tokenizer,
            fr_tokenizer,
        ):
        super().__init__()

        self.dataset = dataset
        self.en_vocab = en_vocab
        self.fr_vocab = fr_vocab
        self.en_tokenizer = en_tokenizer
        self.fr_tokenizer = fr_tokenizer

    def __len__(self):
        """Return the number of examples in the dataset.
        """
        return len(self.dataset)

    def __getitem__(self, index: int) -> tuple:
        """Return a sample.

        Args
        ----
            index: Index of the sample.

        Output
        ------
            en_tokens: English tokens of the sample, as a LongTensor.
            fr_tokens: French tokens of the sample, as a LongTensor.
        """
        # Get the strings
        en_sentence, fr_sentence = self.dataset[index]

        # To list of words
        # We also add the beggining-of-sentence and end-of-sentence tokens
        en_tokens = ['<bos>'] + self.en_tokenizer(en_sentence) + ['<eos>']
        fr_tokens = ['<bos>'] + self.fr_tokenizer(fr_sentence) + ['<eos>']

        # To list of tokens
        en_tokens = self.en_vocab(en_tokens)  # list[int]
        fr_tokens = self.fr_vocab(fr_tokens)

        return torch.LongTensor(en_tokens), torch.LongTensor(fr_tokens)


def yield_tokens(dataset, tokenizer, lang):
    """Tokenize the whole dataset and yield the tokens.
    """
    assert lang in ('en', 'fr')
    sentence_idx = 0 if lang == 'en' else 1

    for sentences in dataset:
        sentence = sentences[sentence_idx]
        tokens = tokenizer(sentence)
        yield tokens


def build_vocab(dataset: list, en_tokenizer, fr_tokenizer, min_freq: int):
    """Return two vocabularies, one for each language.
    """
    en_vocab = build_vocab_from_iterator(
        yield_tokens(dataset, en_tokenizer, 'en'),
        min_freq=min_freq,
        specials=SPECIALS,
    )
    en_vocab.set_default_index(en_vocab['<unk>'])  # Default token for unknown words

    fr_vocab = build_vocab_from_iterator(
        yield_tokens(dataset, fr_tokenizer, 'fr'),
        min_freq=min_freq,
        specials=SPECIALS,
    )
    fr_vocab.set_default_index(fr_vocab['<unk>'])

    return en_vocab, fr_vocab


def preprocess(
        dataset: list,
        en_tokenizer,
        fr_tokenizer,
        max_words: int,
    ) -> list:
    """Preprocess the dataset.
    Remove samples where at least one of the sentences are too long.
    Those samples takes too much memory.
    Also remove the pending '\n' at the end of sentences.
    """
    filtered = []

    for en_s, fr_s in dataset:
        if len(en_tokenizer(en_s)) >= max_words or len(fr_tokenizer(fr_s)) >= max_words:
            continue

        en_s = en_s.replace('\n', '')
        fr_s = fr_s.replace('\n', '')

        filtered.append((en_s, fr_s))

    return filtered


def build_datasets(
        max_sequence_length: int,
        min_token_freq: int,
        en_tokenizer,
        fr_tokenizer,
        train: list,
        val: list,
    ) -> tuple:
    """Build the training, validation and testing datasets.
    It takes care of the vocabulary creation.

    Args
    ----
        - max_sequence_length: Maximum number of tokens in each sequences.
            Having big sequences increases dramatically the VRAM taken during training.
        - min_token_freq: Minimum number of occurences each token must have
            to be saved in the vocabulary. Reducing this number increases
            the vocabularies's size.
        - en_tokenizer: Tokenizer for the english sentences.
        - fr_tokenizer: Tokenizer for the french sentences.
        - train and val: List containing the pairs (english, french) sentences.


    Output
    ------
        - (train_dataset, val_dataset): Tuple of the two TranslationDataset objects.
    """
    datasets = [
        preprocess(samples, en_tokenizer, fr_tokenizer, max_sequence_length)
        for samples in [train, val]
    ]

    en_vocab, fr_vocab = build_vocab(datasets[0], en_tokenizer, fr_tokenizer, min_token_freq)

    datasets = [
        TranslationDataset(samples, en_vocab, fr_vocab, en_tokenizer, fr_tokenizer)
        for samples in datasets
    ]

    return datasets

In [7]:
def generate_batch(data_batch: list, src_pad_idx: int, tgt_pad_idx: int) -> tuple:
    """Add padding to the given batch so that all
    the samples are of the same size.

    Args
    ----
        data_batch: List of samples.
            Each sample is a tuple of LongTensors of varying size.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.

    Output
    ------
        en_batch: Batch of tokens for the padded english sentences.
            Shape of [batch_size, max_en_len].
        fr_batch: Batch of tokens for the padded french sentences.
            Shape of [batch_size, max_fr_len].
    """
    en_batch, fr_batch = [], []
    for en_tokens, fr_tokens in data_batch:
        en_batch.append(en_tokens)
        fr_batch.append(fr_tokens)

    en_batch = pad_sequence(en_batch, padding_value=src_pad_idx, batch_first=True)
    fr_batch = pad_sequence(fr_batch, padding_value=tgt_pad_idx, batch_first=True)
    return en_batch, fr_batch

# Models architecture
This is where you have to code the architectures.

In a machine translation task, the model takes as input the whole
source sentence along with the current known tokens of the target,
and predict the next token in the target sequence.
This means that the target tokens are predicted in an autoregressive
manner, starting from the first token (right after the *\<bos\>* token) and producing tokens one by one until the last *\<eos\>* token.

Formally, we define $s = [s_1, ..., s_{N_s}]$ as the source sequence made of $N_s$ tokens.
We also define $t^i = [t_1, ..., t_i]$ as the target sequence at the beginning of the step $i$.

The output of the model parameterized by $\theta$ is:

$$
T_{i+1} = p(t_{i+1} | s, t^i ; \theta )
$$

Where $T_{i+1}$ is the distribution of the next token $t_{i+1}$.

The loss is simply a *cross entropy loss* over the whole steps, where each class is a token of the vocabulary.

![RNN schema for machinea translation](https://www.simplilearn.com/ice9/free_resources_article_thumb/machine-translation-model-with-encoder-decoder-rnn.jpg)

Note that in this image the english sentence is provided in reverse.

---

In pytorch, there is no dinstinction between an intermediate layer or a whole model having multiple layers in itself.
Every layers or models inherit from the `torch.nn.Module`.
This module needs to define the `__init__` method where you instanciate the layers,
and the `forward` method where you decide how the inputs and the layers of the module interact between them.
Thanks to the autograd computations of pytorch, you do not have
to implement any backward method!

A really important advice is to **always look at
the shape of your input and your output.**
From that, you can often guess how the layers should interact
with the inputs to produce the right output.
You can also easily detect if there's something wrong going on.

You are more than advised to use the `einops` library and the `torch.einsum` function. This will require less operations than 'classical' code, but note that it's a bit trickier to use.
This is a way of describing tensors manipulation with strings, bypassing the multiple tensor methods executed in the background.

Reminders:
You can find a basic presentation of `einops` [here](https://einops.rocks/1-einops-basics/).
Here is a formal academic paper about einops [here](https://openreview.net/pdf?id=oapKSVM2bcj)

**A great tutorial on pytorch can be found [here](https://stanford.edu/class/cs224n/materials/CS224N_PyTorch_Tutorial.html).**
Spending 3 hours on this tutorial is *no* waste of time.

## RNN models

### RNN
Here you have to implement a recurrent neural network. You will need to create a single RNN Layer, and a module allowing to stack these layers. Look up the pytorch documentation to figure out this module's operations and what is communicated from one layer to another


In [8]:
"""
This snippet comes from:
harvardnlp. (3 avril 2018). The Illustrated Transformer.
URL: https://nlp.seas.harvard.edu/2018/04/03/attention.html
"""
import copy
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [9]:
from einops import rearrange
class RNNCell(nn.Module):
    """A single RNN layer.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            dropout: float,
        ):
        super().__init__()

        self.linear_1 = nn.Linear(input_size, hidden_size).to(device)
        self.linear_2 = nn.Linear(hidden_size, hidden_size).to(device)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor) -> tuple:
        """Go through all the sequence in x, iteratively updating
        the hidden state h.

        Args
        ----
            x: Input sequence.
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state.
                Shape of [batch_size, hidden_size].

        Output
        ------
            y: Token embeddings.
                Shape of [batch_size, seq_len, hidden_size].
            h: Last hidden state.
                Shape of [batch_size, hidden_size].
        """
        x = x.transpose(0,1)
        sl = x.size(0)
        bs = x.size(1)
        ins = h.size(1)

        y = torch.zeros(sl, bs, ins).to(device)
        for i, s in enumerate(x):
          h = torch.tanh(self.linear_1(s) + self.linear_2(h))
          h = self.dropout(h)
          y[i] = h

        y = y.transpose(0,1)
        return y, h

class RNN(nn.Module):
    """Implementation of an RNN based
    on https://pytorch.org/docs/stable/generated/torch.nn.RNN.html.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        num_layers: Number of layers (RNNCell or GRUCell).
        dropout: Dropout rate.
        model_type: Either 'RNN' or 'GRU', to select which model we want.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            num_layers: int,
            dropout: float
        ):
      super().__init__()

      self.input_size = input_size
      self.hidden_size = hidden_size
      self.num_layers = num_layers
      self.layers = nn.ModuleList([RNNCell(input_size, hidden_size, dropout)])
      self.layers.extend(clones(RNNCell(hidden_size, hidden_size, dropout), num_layers-1))


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor=None) -> tuple:
      """Pass the input sequence through all the RNN cells.
      Returns the output and the final hidden state of each RNN layer

      Args
      ----
          x: Input sequence.
              Shape of [batch_size, seq_len, input_size].
          h: Hidden state for each RNN layer.
              Can be None, in which case an initial hidden state is created.
              Shape of [batch_size, n_layers, hidden_size].

      Output
      ------
          y: Output embeddings for each token after the RNN layers.
              Shape of [batch_size, seq_len, hidden_size].
          h: Final hidden state.
              Shape of [batch_size, n_layers, hidden_size].
      """
      if h is None:
        h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)

      h_next = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
      h = h.transpose(0,1)

      out = x
      for i, layer in enumerate(self.layers):
        out, h_next[i] = layer(out, h[i])

      h_next = h_next.transpose(0,1)
      y = out
      return y, h_next

### GRU
Here you have to implement a GRU-RNN. This architecture is close to the Vanilla RNN but perform different operations. Look up the pytorch documentation to figure out the differences.

In [10]:

class GRUCell(nn.Module):
    """A single GRU layer.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Initialize six linear transformation layers for the GRU gates:
        # Syntax: self.layer_name = nn.Linear(...).to(device)
        # - linear_i_r: transforms input to reset gate (input_size → hidden_size)
        self.linear_i_r = nn.Linear(input_size,  hidden_size).to(device)

        # - linear_h_r: transforms hidden state to reset gate (hidden_size → hidden_size)
        self.linear_h_r = nn.Linear(hidden_size, hidden_size).to(device)

        # - linear_i_z: transforms input to update gate (input_size → hidden_size)
        self.linear_i_z = nn.Linear(input_size,  hidden_size).to(device)

        # - linear_h_z: transforms hidden state to update gate (hidden_size → hidden_size)
        self.linear_h_z = nn.Linear(hidden_size, hidden_size).to(device)

        # - linear_i_n: transforms input to new gate/candidate (input_size → hidden_size)
        self.linear_i_n = nn.Linear(input_size,  hidden_size).to(device)

        # - linear_h_n: transforms hidden state to new gate (hidden_size → hidden_size)
        self.linear_h_n = nn.Linear(hidden_size, hidden_size).to(device)

        # Then create a dropout layer: self.dropout = nn.Dropout(dropout)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor) -> tuple:
        """
        Args
        ----
            x: Input sequence.
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state.
                Shape of [batch_size, hidden_size].

        Output
        ------
            y: Token embeddings.
                Shape of [batch_size, seq_len, hidden_size].
            h: Last hidden state.
                Shape of [batch_size, hidden_size].
        """

        # TODO
        # Implement the GRU forward pass:
        # 1. Transpose input from [batch, seq_len, input_size] to [seq_len, batch, input_size]
        #    using x = x.transpose(0,1)
        x = x.transpose(0,1)

        # 2. Extract dimensions: sl = x.size(0), bs = x.size(1), ins = h.size(1)
        sl = x.size(0)
        bs = x.size(1)
        ins = h.size(1)

        # 3. Initialize output tensor: y = torch.zeros(sl, bs, ins).to(device)
        y = torch.zeros(sl, bs, ins).to(device)

        # 4. Loop over sequence: for i, s in enumerate(x):
        for i, s in enumerate(x):
        # 5. Compute reset gate: r = torch.sigmoid(self.linear_i_r(s) + self.linear_h_r(h))
          r = torch.sigmoid(self.linear_i_r(s) + self.linear_h_r(h))

        # 6. Compute update gate: z = ...
          z = torch.sigmoid(self.linear_i_z(s) + self.linear_h_z(h))

        # 7. Compute candidate: n = ...
          n = torch.tanh(self.linear_i_n(s) + r * self.linear_h_n(h))

        # 8. Update hidden state:
          h = (1.0 - z) * n + z * h

        # 9. Apply dropout: h = ...
          h = self.dropout(h)

        # 10. Store in output: y[i] = h
          y[i] = h

        # 11. Return transposed y and final h
        y = y.transpose(0,1)
        return y, h


class GRU(nn.Module):
    """Implementation of a GRU based on https://pytorch.org/docs/stable/generated/torch.nn.GRU.html.

    Parameters
    ----------
        input_size: Size of each input token.
        hidden_size: Size of each RNN hidden state.
        num_layers: Number of layers.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            input_size: int,
            hidden_size: int,
            num_layers: int,
            dropout: float,
        ):

        super().__init__()

        # TODO
        # Store dimensions as instance variables
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Create a ModuleList to hold all GRU layers:
        # - First layer: GRUCell(input_size, hidden_size, dropout)
        # Syntax: self.layers = nn.ModuleList([GRUCell(...)])
        self.layers = nn.ModuleList([GRUCell(input_size, hidden_size, dropout)])

        # - Remaining layers: GRUCell(hidden_size, hidden_size, dropout) × (num_layers-1)
        # Then extend: self.layers.extend(clones(GRUCell(hidden_size, hidden_size, dropout), num_layers-1))
        self.layers.extend(clones(GRUCell(hidden_size, hidden_size, dropout), num_layers-1))



    def forward(self, x: torch.FloatTensor, h: torch.FloatTensor=None) -> tuple:
        """
        Args
        ----
            x: Input sequence
                Shape of [batch_size, seq_len, input_size].
            h: Initial hidden state for each layer.
                If 'None', then an initial hidden state (a zero filled tensor)
                is created.
                Shape of [batch_size, n_layers, hidden_size].

        Output
        ------
            output:
                Shape of [batch_size, seq_len, hidden_size].
            h_n: Final hidden state.
                Shape of [batch_size, n_layers, hidden size].
        """

        # TODO
        # Implement multi-layer GRU forward pass:
        # 1. Initialize hidden state if None:
        #    if h == None:  h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)
        #    Shape: [batch_size, num_layers, hidden_size]
        if h == None:
          h = torch.zeros(x.size(0), self.num_layers, self.hidden_size).to(device)

        # 2. Create tensor for next hidden states:
        #    h_next = torch.zeros...
        #    Shape: [num_layers, batch_size, hidden_size]
        h_next = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)

        # 3. Transpose h from [batch, layers, hidden] to [layers, batch, hidden]:
        h = h.transpose(0, 1)

        # 4. Initialize output with input: out = x
        out = x

        # 5. Loop through each GRU layer sequentially:
        #    for i, layer in enumerate(self.layers):
        #        out, h_next[i] = layer(out, h[i])
        #    Each layer takes output from previous layer and its corresponding hidden state
        for i, layer in enumerate(self.layers):
          out, h_next[i] = layer(out, h[i])

        # 6. Transpose h_next back to [batch, layers, hidden]:
        #    h_next = h_next.transpose(0,1)
        h_next = h_next.transpose(0, 1)

        # 7. Assign final output: y = out
        y = out

        # 8. Return (final layer output) and (all layers' hidden states)
        return y, h_next



### Translation RNN

This module instanciates a vanilla RNN or a GRU-RNN and performs the translation task. You have to:
* Encode the source sequence
* Pass the final hidden state of the encoder to the decoder(one for each layer)
* Decode the hidden state into the target sequence

We use teacher forcing for training, meaning that when the next token is predicted, that prediction is based on the previous true target tokens.

In [11]:
class TranslationRNN(nn.Module):
    """Basic RNN encoder and decoder for a translation task.
    It can run as a vanilla RNN or a GRU-RNN.

    Parameters
    ----------
        n_tokens_src: Number of tokens in the source vocabulary.
        n_tokens_tgt: Number of tokens in the target vocabulary.
        dim_embedding: Dimension size of the word embeddings (for both language).
        dim_hidden: Dimension size of the hidden layers in the RNNs
            (for both the encoder and the decoder).
        n_layers: Number of layers in the RNNs.
        dropout: Dropout rate.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.
        model_type: Either 'RNN' or 'GRU', to select which model we want.
    """

    def __init__(
            self,
            n_tokens_src: int,
            n_tokens_tgt: int,
            dim_embedding: int,
            dim_hidden: int,
            n_layers: int,
            dropout: float,
            src_pad_idx: int,
            tgt_pad_idx: int,
            model_type: str
        ):
        super().__init__()

        self.src_embeddings = nn.Embedding(n_tokens_src, dim_embedding, src_pad_idx)
        self.tgt_embeddings = nn.Embedding(n_tokens_tgt, dim_embedding, tgt_pad_idx)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

        Model = RNN if model_type == "RNN" else GRU

        self.encoder = Model(dim_embedding, dim_hidden, n_layers, dropout)
        self.norm = nn.LayerNorm(dim_hidden)
        self.decoder = Model(dim_embedding, dim_hidden, n_layers, dropout)

        self.seq = nn.Sequential(
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, dim_hidden),
            nn.LeakyReLU(),
            nn.LayerNorm(dim_hidden),
            nn.Linear(dim_hidden, n_tokens_tgt)
        )

    def forward(
        self,
        source: torch.LongTensor,
        target: torch.LongTensor
    ) -> torch.FloatTensor:
        """Predict the source tokens based on the target tokens.

        Args
        ----
            source: Batch of source sentences.
                Shape of [batch_size, src_seq_len].
            target: Batch of target sentences.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y: Batch of predictions where each token is the prediction of
                the next token in the sentence.
                Shape of [batch_size, tgt_seq_len].
        """

        # Reversing the source sequences
        source = torch.fliplr(source)

        src_emb = self.src_embeddings(source)
        out, hidden = self.encoder(src_emb)

        hidden = self.norm(hidden)

        tgt_emb = self.tgt_embeddings(target)
        y, hidden = self.decoder(tgt_emb, hidden)

        y = self.seq(y)

        return y



## Transformer model
Here you have to code the Transformer architecture.
It is divided in three parts:
* Attention layers
* Encoder and decoder layers
* Main layers (gather the encoder and decoder layers)

The [illustrated transformer](https://jalammar.github.io/illustrated-transformer/) blog can help you
understanding how the architecture works.
Once this is done, you can use [the annontated transformer](https://nlp.seas.harvard.edu/2018/04/03/attention.html) to have an idea of how to code this architecture.
We encourage you to use `torch.einsum` and the `einops` library as much as you can. It will make your code simpler.

---
**Implementation order**

To help you with the implementation, we advise you following this order:
* Implement `TranslationTransformer` and use `nn.Transformer` instead of `Transformer`
* Implement `Transformer` and use `nn.TransformerDecoder` and `nn.TransformerEnocder`
* Implement the `TransformerDecoder` and `TransformerEncoder` and use `nn.MultiHeadAttention`
* Implement `MultiHeadAttention`

Do not forget to add `batch_first=True` when necessary in the `nn` modules.

### Attention layers
We use a `MultiHeadAttention` module, that is able to perform self-attention aswell as cross-attention (depending on what you give as queries, keys and values).

**Attention**


It takes the multiheaded queries, keys and values as input.
It computes the attention between the queries and the keys and return the attended values.

The implementation of this function can greatly be improved with *einsums*.

**MultiheadAttention**

Computes the multihead queries, keys and values and feed them to the `attention` function.
You also need to merge the key padding mask and the attention mask into one mask.

The implementation of this module can greatly be improved with *einops.rearrange*.

In [12]:
from einops.layers.torch import Rearrange
from einops import rearrange
import math
import torch.nn.functional as F
import copy

def attention(
        q: torch.FloatTensor,
        k: torch.FloatTensor,
        v: torch.FloatTensor,
        mask: torch.BoolTensor=None,
        dropout: nn.Dropout=None,
    ) -> tuple:
    """Computes multihead scaled dot-product attention from the
    projected queries, keys and values.

    Args
    ----
        q: Batch of queries.
            Shape of [batch_size, seq_len_1, n_heads, dim_model].
        k: Batch of keys.
            Shape of [batch_size, seq_len_2, n_heads, dim_model].
        v: Batch of values.
            Shape of [batch_size, seq_len_2, n_heads, dim_model].
        mask: Prevent tokens to attend to some other tokens (for padding or autoregressive attention).
            Attention is prevented where the mask is `True`.
            Shape of [batch_size, n_heads, seq_len_1, seq_len_2],
            or broadcastable to that shape.
        dropout: Dropout layer to use.

    Output
    ------
        y: Multihead scaled dot-attention between the queries, keys and values.
            Shape of [batch_size, seq_len_1, n_heads, dim_model].
        attn: Computed attention mask.
            Shape of [batch_size, n_heads, seq_len_1, seq_len_2].
    """
    # TODO
    # Implement scaled dot-product attention:
    # 1. Get dimension of each head: dim_model = q.size(-1)
    dim_model = q.size(-1)

    # 2. Compute attention scores using Einstein summation:
    #    attn = torch.einsum(...) / sqrt(model dimension)
    #    This should compute Q·K^T / sqrt(d_k) where:
    #    - b=batch, h=heads, l=query_seq_len, t=key_seq_len, k=dim_per_head
    attn = torch.einsum('bhlk, bhtk -> bhlt', q, k) / math.sqrt(dim_model)

    # 3. Apply mask if provided: attn = attn.masked_fill(mask==0, -math.inf)
    #    This sets masked positions to -inf so softmax gives them 0 weight
    if mask is not None:
      attn = attn.masked_fill(mask==0, -math.inf)

    # 4. Apply softmax: attn = torch.softmax(attn, dim=-1)
    attn = torch.softmax(attn, dim=-1)

    # 5. Apply dropout if provided: attn = dropout(attn)
    if dropout is not None:
      attn = dropout(attn)

    # 6. Compute weighted sum of values: y = torch.einsum(..., [attn, v])
    y = torch.einsum('bhlt, bhtk -> bhlk', [attn, v])

    # 7. Return both the output y and attention weights attn
    return y, attn

    # See the following references:
    # Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
    # Lippe, Phillipe. (2022). Tutorial 6: Transformers and Multi-Head Attention. https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html


class MultiheadAttention(nn.Module):
    """Multihead attention module.
    Can be used as a self-attention and cross-attention layer.
    The queries, keys and values are projected into multiple heads
    before computing the attention between those tensors.

    Parameters
    ----------
        dim: Dimension of the input tokens.
        n_heads: Number of heads. `dim` must be divisible by `n_heads`.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            dim: int,
            n_heads: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Store attention parameters (self.):
        # dim = (total embedding dimension)
        self.dim = dim

        # n_heads = (number of attention heads)
        self.n_heads = n_heads

        # d_k = (dimension per head)
        self.d_k = dim // n_heads

        # Create three linear projection layers (no bias):
        # - self.q_linear = nn.Linear(dim, dim)  for queries
        self.q_linear = nn.Linear(dim, dim, bias=False)

        # - self.k_linear = ... for keys
        self.k_linear = nn.Linear(dim, dim, bias=False)

        # - self.v_linear = ... for values
        self.v_linear = nn.Linear(dim, dim, bias=False)

        # Create a dropout layer
        self.dropout = nn.Dropout(dropout)

        # Create output projection: self.out = ...
        self.out = nn.Linear(dim, dim, bias=False)

        # See the following for help:
        # Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec



    def forward(
            self,
            q: torch.FloatTensor,
            k: torch.FloatTensor,
            v: torch.FloatTensor,
            key_padding_mask: torch.BoolTensor = None,
            attn_mask: torch.BoolTensor = None,
        ) -> torch.FloatTensor:
        """Computes the scaled multi-head attention form the input queries,
        keys and values.

        Project those queries, keys and values before feeding them
        to the `attention` function.

        The masks are boolean masks. Tokens are prevented to attends to
        positions where the mask is `True`.

        Args
        ----
            q: Batch of queries.
                Shape of [batch_size, seq_len_1, dim_model].
            k: Batch of keys.
                Shape of [batch_size, seq_len_2, dim_model].
            v: Batch of values.
                Shape of [batch_size, seq_len_2, dim_model].
            key_padding_mask: Prevent attending to padding tokens.
                Shape of [batch_size, seq_len_2].
            attn_mask: Prevent attending to subsequent tokens.
                Shape of [seq_len_1, seq_len_2].

        Output
        ------
            y: Computed multihead attention.
                Shape of [batch_size, seq_len_1, dim_model].
        """
        # TODO
        # Implement multi-head attention forward pass:
        # 1. Get batch_size = q.size(0)
        batch_size = q.size(0)

        # 2. Handle None masks by creating all-ones tensors:
        #    if key_padding_mask is None: key_padding_mask = torch.ones((batch_size, k.size(1))).to(device)
        if key_padding_mask is None:
          key_padding_mask = torch.ones((batch_size, k.size(1))).to(device)

        #    if attn_mask is None: attn_mask = torch.ones((q.size(1), k.size(1))).to(device)
        if attn_mask is None:
          attn_mask = torch.ones((q.size(1), k.size(1))).to(device)

        # 3. Combine masks: mask = attn_mask.logical_and(key_padding_mask).unsqueeze(1)
        mask = attn_mask.logical_and(key_padding_mask).unsqueeze(1)

        # 4. Project and reshape Q, K, V using einops rearrange:
        #    q = rearrange(self.q_linear(q), 'b s (hnum k) -> b hnum s k', hnum=self.n_heads)
        q = rearrange(self.q_linear(q), 'b s (hnum k) -> b hnum s k', hnum=self.n_heads)

        #    k = rearrange(...)
        k = rearrange(self.k_linear(k), 'b s (hnum k) -> b hnum s k', hnum=self.n_heads)

        #    v = rearrange(...)
        v = rearrange(self.v_linear(v), 'b s (hnum k) -> b hnum s k', hnum=self.n_heads)

        # 5. Compute attention: y, attn = attention(q, k, v, mask, self.dropout)
        y, attn = attention(q, k, v, mask, self.dropout)

        # 6. Merge heads: y = rearrange(y, 'b hnum s v -> b s (hnum v)')
        y = rearrange(y, 'b hnum s v -> b s (hnum v)')

        # 7. Apply output projection: y = self.out(y)
        y = self.out(y)

        # 8. Return y
        return y

        # See: Lynn-Evans, Samuel. (2018). How to code The Transformer in Pytorch. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
        # Einops. (unspecified). Writing a better code with pytorch and einops. https://einops.rocks/pytorch-examples.html



### Positional Encoding

In [13]:
class PositionalEncoding(nn.Module):
    """
    This PE module comes from:
    Pytorch. (2021). LANGUAGE MODELING WITH NN.TRANSFORMER AND TORCHTEXT. https://pytorch.org/tutorials/beginner/transformer_tutorial.html
    """
    def __init__(self, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len).unsqueeze(1).to(device)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)).to(device)
        pe = torch.zeros(max_len, 1, d_model).to(device)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = rearrange(x, "b s e -> s b e")
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        x = x + self.pe[:x.size(0)]
        x = rearrange(x, "s b e -> b s e")
        return self.dropout(x)

### Encoder and decoder layers

**TranformerEncoder**

Apply self-attention layers onto the source tokens.
It only needs the source key padding mask.


**TranformerDecoder**

Apply masked self-attention layers to the target tokens and cross-attention
layers between the source and the target tokens.
It needs the source and target key padding masks, and the target attention mask.

In [14]:
class FeedForward(nn.Module):
    """
    Simple feedforward module used in the transformer model
    Inspired from: https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec
    """
    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float
      ):
      super().__init__()

      self.linear_1 = nn.Linear(d_model, d_ff)
      self.linear_2 = nn.Linear(d_ff, d_model)
      self.dropout_1 = nn.Dropout(dropout)
      self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x):
      x = self.linear_1(x)
      x = F.relu(x)
      x = self.dropout_1(x)
      x = self.linear_2(x)
      x = self.dropout_2(x)
      return x


class TransformerDecoderLayer(nn.Module):
    """Single decoder layer.

    Parameters
    ----------
        d_model: The dimension of decoders inputs/outputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            nhead: int,
            dropout: float
        ):
        super().__init__()

        # TODO
        # Initialize decoder layer components:
        # 1. Masked self-attention: self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        #    (target attends to previous target tokens only)
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)

        # 2. Cross-attention: self.enc_dec_attn = MultiheadAttention(d_model, nhead, dropout)
        #    (target queries attend to encoder output keys/values)
        self.enc_dec_attn = MultiheadAttention(d_model, nhead, dropout)

        # 3. Feedforward network: self.ff = FeedForward(d_model, d_ff, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)

        # 4. Dropout layers: self.dropout_1 = nn.Dropout(dropout), self.dropout_2 = nn.Dropout(dropout)
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

        # 5. Layer normalization: self.norm_1, self.norm_2, self.norm_3 = nn.LayerNorm(d_model)
        #self.norm_1, self.norm_2, self.norm_3 = nn.LayerNorm(d_model) ??
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)
        self.norm_3 = nn.LayerNorm(d_model)



    def forward(
            self,
            tgt: torch.FloatTensor,
            memory: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            memory_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor,
        ) -> torch.FloatTensor:
        """Decode the next target tokens based on the previous tokens.

        Args
        ----
            src: Batch of source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of target sentences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            memory_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y:  Batch of sequence of embeddings representing the predicted target tokens
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        x = tgt
        x = self.norm_1(x + self.dropout_1(self.self_attn(x, x, x, attn_mask=tgt_mask_attn, key_padding_mask=tgt_key_padding_mask)))
        x = self.norm_2(x + self.dropout_2(self.enc_dec_attn.forward(x, memory, memory, key_padding_mask=memory_key_padding_mask)))
        x = self.norm_3(x + self.ff(x))

        return x


class TransformerDecoder(nn.Module):
    """Implementation of the transformer decoder stack.

    Parameters
    ----------
        d_model: The dimension of decoders inputs/outputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        num_decoder_layers: Number of stacked decoders.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            num_decoder_layer:int ,
            nhead: int,
            dropout: float
        ):
        super().__init__()

        decoder_layer = TransformerDecoderLayer(d_model, d_ff, nhead, dropout)
        self.decoder_layers = clones(decoder_layer, num_decoder_layer)

    def forward(
            self,
            tgt: torch.FloatTensor,
            memory: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            memory_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor,
        ) -> torch.FloatTensor:
        """Decodes the source sequence by sequentially passing.
        the encoded source sequence and the target sequence through the decoder stack.

        Args
        ----
            src: Batch of encoded source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of taget sentences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            memory_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y:  Batch of sequence of embeddings representing the predicted target tokens
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        out = tgt
        for layer in self.decoder_layers:
          out = layer(out, memory, tgt_mask_attn=tgt_mask_attn, tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
        return out


class TransformerEncoderLayer(nn.Module):
    """Single encoder layer.

    Parameters
    ----------
        d_model: The dimension of input tokens.
        dim_feedforward: Hidden dimension of the feedforward networks.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            d_ff: int,
            nhead: int,
            dropout: float,
        ):
        super().__init__()

        # TODO
        # Initialize encoder layer components:
        # 1. Self-attention: self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)

        # 2. Position-wise feedforward: self.ff = FeedForward(d_model, d_ff, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)

        # 3. Dropout layer: self.dropout = nn.Dropout(dropout)
        self.dropout = nn.Dropout(dropout)

        # 4. Layer normalization: self.norm_1 = nn.LayerNorm(d_model), self.norm_2 = nn.LayerNorm(d_model)
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)


    def forward(
        self,
        src: torch.FloatTensor,
        key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Encodes the input. Does not attend to masked inputs.

        Args
        ----
            src: Batch of embedded source tokens.
                Shape of [batch_size, src_seq_len, dim_model].
            key_padding_mask: Mask preventing attention to padding tokens.
                Shape of [batch_size, src_seq_len].

        Output
        ------
            y: Batch of encoded source tokens.
                Shape of [batch_size, src_seq_len, dim_model].
        """
        x = src
        x = self.norm_1(x + self.dropout(self.self_attn.forward(x, x, x, key_padding_mask=key_padding_mask)))
        x = self.norm_2(x + self.ff(x))
        return x


class TransformerEncoder(nn.Module):
    """Implementation of the transformer encoder stack.

    Parameters
    ----------
        d_model: The dimension of encoders inputs.
        dim_feedforward: Hidden dimension of the feedforward networks.
        num_encoder_layers: Number of stacked encoders.
        nheads: Number of heads for each multi-head attention.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            dim_feedforward: int,
            num_encoder_layers: int,
            nheads: int,
            dropout: float
        ):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(d_model, dim_feedforward, nheads, dropout)
        self.encoder_layers = clones(encoder_layer, num_encoder_layers)


    def forward(
            self,
            src: torch.FloatTensor,
            key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Encodes the source sequence by sequentially passing.
        the source sequence through the encoder stack.

        Args
        ----
            src: Batch of embedded source sentences.
                Shape of [batch_size, src_seq_len, dim_model].
            key_padding_mask: Mask preventing attention to padding tokens.
                Shape of [batch_size, src_seq_len].

        Output
        ------
            y: Batch of encoded source sequence.
                Shape of [batch_size, src_seq_len, dim_model].
        """
        out = src
        for layer in self.encoder_layers:
          out = layer(out, key_padding_mask=key_padding_mask)
        return out

### Main layers
This section gather the `Transformer` and the `TranslationTransformer` modules.

**Transformer**


The classical transformer architecture.
It takes the source and target tokens embeddings and
do the forward pass through the encoder and decoder.

**Translation Transformer**

Compute the source and target tokens embeddings, and apply a final head to produce next token logits.
The output must not be the softmax but just the logits, because we use the `nn.CrossEntropyLoss`.

It also creates the *src_key_padding_mask*, the *tgt_key_padding_mask* and the *tgt_mask_attn*.

In [15]:
import math
class Transformer(nn.Module):
    """Implementation of a Transformer based on the paper: https://arxiv.org/pdf/1706.03762.pdf.

    Parameters
    ----------
        d_model: The dimension of encoders/decoders inputs/ouputs.
        nhead: Number of heads for each multi-head attention.
        num_encoder_layers: Number of stacked encoders.
        num_decoder_layers: Number of stacked encoders.
        dim_feedforward: Hidden dimension of the feedforward networks.
        dropout: Dropout rate.
    """

    def __init__(
            self,
            d_model: int,
            nhead: int,
            num_encoder_layers: int,
            num_decoder_layers: int,
            dim_feedforward: int,
            dropout: float,
        ):
        super().__init__()

        self.d_model = d_model
        self.encoder = TransformerEncoder(d_model, dim_feedforward, num_encoder_layers, nhead, dropout)
        self.decoder = TransformerDecoder(d_model, dim_feedforward, num_decoder_layers, nhead, dropout)

    def forward(
            self,
            src: torch.FloatTensor,
            tgt: torch.FloatTensor,
            tgt_mask_attn: torch.BoolTensor,
            src_key_padding_mask: torch.BoolTensor,
            tgt_key_padding_mask: torch.BoolTensor
        ) -> torch.FloatTensor:
        """Compute next token embeddings.

        Args
        ----
            src: Batch of source sequences.
                Shape of [batch_size, src_seq_len, dim_model].
            tgt: Batch of target sequences.
                Shape of [batch_size, tgt_seq_len, dim_model].
            tgt_mask_attn: Mask to prevent attention to subsequent tokens.
                Shape of [tgt_seq_len, tgt_seq_len].
            src_key_padding_mask: Mask to prevent attention to padding in src sequence.
                Shape of [batch_size, src_seq_len].
            tgt_key_padding_mask: Mask to prevent attention to padding in tgt sequence.
                Shape of [batch_size, tgt_seq_len].

        Output
        ------
            y: Next token embeddings, given the previous target tokens and the source tokens.
                Shape of [batch_size, tgt_seq_len, dim_model].
        """
        out = self.encoder.forward(src, src_key_padding_mask)
        out = self.decoder.forward(tgt, out, tgt_mask_attn, src_key_padding_mask, tgt_key_padding_mask)
        return out


class TranslationTransformer(nn.Module):
    """Basic Transformer encoder and decoder for a translation task.
    Manage the masks creation, and the token embeddings.
    Position embeddings can be learnt with a standard `nn.Embedding` layer.

    Parameters
    ----------
        n_tokens_src: Number of tokens in the source vocabulary.
        n_tokens_tgt: Number of tokens in the target vocabulary.
        n_heads: Number of heads for each multi-head attention.
        dim_embedding: Dimension size of the word embeddings (for both language).
        dim_hidden: Dimension size of the feedforward layers
            (for both the encoder and the decoder).
        n_layers: Number of layers in the encoder and decoder.
        dropout: Dropout rate.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.
    """
    def __init__(
            self,
            n_tokens_src: int,
            n_tokens_tgt: int,
            n_heads: int,
            dim_embedding: int,
            dim_hidden: int,
            n_layers: int,
            dropout: float,
            src_pad_idx: int,
            tgt_pad_idx: int,
        ):
        super().__init__()

        self.dim_embedding = dim_embedding
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        self.pe_src = PositionalEncoding(dim_embedding, dropout)
        self.pe_tgt = PositionalEncoding(dim_embedding, dropout)

        self.embedding_src = nn.Embedding(n_tokens_src, dim_embedding)
        self.embedding_tgt = nn.Embedding(n_tokens_tgt, dim_embedding)

        self.my_transformer = Transformer(dim_embedding, n_heads, n_layers, n_layers, dim_hidden, dropout)
        self.out_layer = nn.Linear(dim_embedding, n_tokens_tgt)


    def forward(
            self,
            source: torch.LongTensor,
            target: torch.LongTensor
        ) -> torch.FloatTensor:
        """Predict the target tokens based on the source tokens.

        Args
        ----
            source: Batch of source sentences.
                Shape of [batch_size, seq_len_src].
            target: Batch of target sentences.
                Shape of [batch_size, seq_len_tgt].

        Output
        ------
            y: Batch of predictions of the next token distributions in the target sentences.
                Shape of [batch_size, seq_len_tgt, n_tokens_tgt].
        """
        # TODO
        # Implement full translation forward pass:
        # 1. Build causal mask for target: tgt_mask = self.build_mask(target.size(1))
        #    This creates upper triangular mask preventing attention to future positions
        tgt_mask = self.build_mask(target.size(1))

        # 2. Build padding masks:
        #    src_key_padding_mask = self.build_pad_mask(source, self.src_pad_idx)
        src_key_padding_mask = self.build_pad_mask(source, self.src_pad_idx)

        #    tgt_key_padding_mask = self.build_pad_mask(target, self.tgt_pad_idx)
        tgt_key_padding_mask = self.build_pad_mask(target, self.tgt_pad_idx)

        # 3. Embed and scale source tokens:
        #    source = self.embedding_src(source) * Scaling by sqrt(d_model) as per original Transformer paper
        source = self.embedding_src(source) * math.sqrt(self.dim_embedding)

        # 4. Add positional encoding to source: source = self.pe_src(source)
        source = self.pe_src(source)

        # 5. Embed and scale target: target = self.embedding_tgt(target) * math.sqrt(self.dim_embedding)
        target = self.embedding_tgt(target) * math.sqrt(self.dim_embedding)

        # 6. Add positional encoding to target: target = self.pe_tgt(target)
        target = self.pe_tgt(target)

        # 7. Pass through transformer:
        #    trans_out = self.my_transformer(src=source, tgt=target, tgt_mask_attn=tgt_mask,
        #                                     src_key_padding_mask=src_key_padding_mask,
        #                                     tgt_key_padding_mask=tgt_key_padding_mask)
        trans_out = self.my_transformer(src=source, tgt=target, tgt_mask_attn=tgt_mask, src_key_padding_mask=src_key_padding_mask, tgt_key_padding_mask=tgt_key_padding_mask)

        # 8. Project to vocabulary: y = self.out_layer(trans_out)
        y = self.out_layer(trans_out)

        # 9. Return y with shape [batch_size, seq_len_tgt, n_tokens_tgt]
        return y



    def build_mask(self, size):
        """
        From:
        MELCHOR, Daniel. (2021). A detailed guide to PyTorch’s nn.Transformer() module. https://towardsdatascience.com/a-detailed-guide-to-pytorchs-nn-transformer-module-c80afbc9ffb1
        """
        attn_shape = (1, size, size)
        mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
        return (torch.from_numpy(mask) == 0).to(device)

    def build_pad_mask(self, tensor, pad_idx):
        mask = (tensor != pad_idx).unsqueeze(-2)
        return mask


# Greedy search

Here you have to implement a geedy search to generate a target translation from a trained model and an input source string.
The next token will simply be the most probable one.

In [16]:
def indices_terminated(
        target: torch.FloatTensor,
        eos_token: int
    ) -> tuple:
    """Split the target sentences between the terminated and the non-terminated
    sentence. Return the indices of those two groups.

    Args
    ----
        target: The sentences.
            Shape of [batch_size, n_tokens].
        eos_token: Value of the End-of-Sentence token.

    Output
    ------
        terminated: Indices of the terminated sentences (who's got the eos_token).
            Shape of [n_terminated, ].
        non-terminated: Indices of the unfinished sentences.
            Shape of [batch_size-n_terminated, ].
    """
    terminated = [i for i, t in enumerate(target) if eos_token in t]
    non_terminated = [i for i, t in enumerate(target) if eos_token not in t]
    return torch.LongTensor(terminated), torch.LongTensor(non_terminated)

In [17]:
def greedy_search(
        model: nn.Module,
        source: str,
        src_vocab: Vocab,
        tgt_vocab: Vocab,
        src_tokenizer,
        device: str,
        max_sentence_length: int,
    ) -> str:
    """Do a greedy search to produce probable translations.

    Args
    ----
        model: The translation model. Assumes it produces logits score (before softmax).
        source: The sentence to translate.
        src_vocab: The source vocabulary.
        tgt_vocab: The target vocabulary.
        device: Device to which we make the inference.
        max_sentence_length: Maximum number of tokens for the translated sentence.

    Output
    ------
        sentence: The translated source sentence.
    """

    src_tokens = ['<bos>'] + src_tokenizer(source) + ['<eos>']
    src_tokens = src_vocab(src_tokens)

    tgt_tokens = ['<bos>']
    tgt_tokens = tgt_vocab(tgt_tokens)

    # To tensor and add unitary batch dimension
    src_tokens = torch.LongTensor(src_tokens).to(device)
    tgt_tokens = torch.LongTensor(tgt_tokens).unsqueeze(dim=0).to(device)
    target_probs = torch.FloatTensor([1]).to(device)
    model.to(device)
    EOS_IDX = tgt_vocab['<eos>']

    with torch.no_grad():
      while tgt_tokens.shape[1] < max_sentence_length:

            batch_size, n_tokens = tgt_tokens.shape

            src = einops.repeat(src_tokens, 't -> b t', b=tgt_tokens.shape[0])
            predicted = model.forward(src, tgt_tokens)
            predicted = torch.softmax(predicted, dim=-1)

            probs, predicted = predicted[:, -1].topk(k=1, dim=-1)

            # Separe between terminated sentences and the others
             # Separe between terminated sentences and the others
            idx_terminated, idx_not_terminated = indices_terminated(tgt_tokens, EOS_IDX)
            idx_terminated, idx_not_terminated = idx_terminated.to(device), idx_not_terminated.to(device)

            tgt_terminated = torch.index_select(tgt_tokens, dim=0, index=idx_terminated)
            tgt_probs_terminated = torch.index_select(target_probs, dim=0, index=idx_terminated)

            filter_t = lambda t: torch.index_select(t, dim=0, index=idx_not_terminated)
            tgt_others = filter_t(tgt_tokens)
            if tgt_others.shape[0] == 0:
              break
            tgt_probs_others = filter_t(target_probs)
            predicted = filter_t(predicted)
            probs = filter_t(probs)

            # Add padding to terminated target
            padd = torch.zeros((len(tgt_terminated), 1), dtype=torch.long, device=device)
            tgt_terminated = torch.cat(
                (tgt_terminated, padd),
                dim=1
            )

            # Update each target sentence probabilities

            tgt_probs_others *= probs.flatten()
            tgt_probs_terminated *= 0.999  # Penalize short sequences overtime

            # Group up the terminated and the others
            target_probs = torch.cat(
                (tgt_probs_others, tgt_probs_terminated),
                dim=0
            )

            n_tokens = tgt_others.shape[1]

            tgt_others = torch.cat((tgt_others, predicted), dim=-1)
            tgt_others = tgt_others.view(batch_size, n_tokens+1)


            tgt_tokens = torch.cat(
                (tgt_others, tgt_terminated),
                dim=0
            )


            target_probs, indices = target_probs.topk(k=1, dim=0)
            tgt_tokens = torch.index_select(tgt_tokens, dim=0, index=indices)


    sentences = []
    for tgt_sentence in tgt_tokens:
        tgt_sentence = list(tgt_sentence)[1:]  # Remove <bos> token
        tgt_sentence = list(takewhile(lambda t: t != EOS_IDX, tgt_sentence))
        tgt_sentence = ' '.join(tgt_vocab.lookup_tokens(tgt_sentence))
        sentences.append(tgt_sentence)

    sentences = [beautify(s) for s in sentences]

    # Join the sentences with their likelihood
    sentences = [(s, p.item()) for s, p in zip(sentences, target_probs)]
    # Sort the sentences by their likelihood
    sentences = [(s, p) for s, p in sorted(sentences, key=lambda k: k[1], reverse=True)]

    return sentences



# Beam search
Beam search is a smarter way of producing a sequence of tokens from
an autoregressive model than just using a greedy search.

The greedy search always choose the most probable token as the unique
and only next target token, and repeat this processus until the *\<eos\>* token is predicted.

Instead, the beam search selects the k-most probable tokens at each step.
From those k tokens, the current sequence is duplicated k times and the k tokens are appended to the k sequences to produce new k sequences.

*You don't have to understand this code, but understanding this code once the TP is over could improve your torch tensors skills.*

---

**More explanations**

Since it is done at each step, the number of sequences grows exponentially (k sequences after the first step, k² sequences after the second...).
In order to keep the number of sequences low, we remove sequences except the top-s most likely sequences.
To do that, we keep track of the likelihood of each sequence.

Formally, we define $s = [s_1, ..., s_{N_s}]$ as the source sequence made of $N_s$ tokens.
We also define $t^i = [t_1, ..., t_i]$ as the target sequence at the beginning of the step $i$.

The output of the model parameterized by $\theta$ is:

$$
T_{i+1} = p(t_{i+1} | s, t^i ; \theta )
$$

Where $T_{i+1}$ is the distribution of the next token $t_{i+1}$.

Then, we define the likelihood of a target sentence $t = [t_1, ..., t_{N_t}]$ as:

$$
L(t) = \prod_{i=1}^{N_t - 1} p(t_{i+1} | s, t_{i}; \theta )
$$

Pseudocode of the beam search:
```
source: [N_s source tokens]  # Shape of [total_source_tokens]
target: [1, <bos> token]  # Shape of [n_sentences, current_target_tokens]
target_prob: [1]  # Shape of [n_sentences]
# We use `n_sentences` as the batch_size dimension

while current_target_tokens <= max_target_length:
    source = repeat(source, n_sentences)  # Shape of [n_sentences, total_source_tokens]
    predicted = model(source, target)[:, -1]  # Predict the next token distributions of all the n_sentences
    tokens_idx, tokens_prob = topk(predicted, k)

    # Append the `n_sentences * k` tokens to the `n_sentences` sentences
    target = repeat(target, k)  # Shape of [n_sentences * k, current_target_tokens]
    target = append_tokens(target, tokens_idx)  # Shape of [n_sentences * k, current_target_tokens + 1]

    # Update the sentences probabilities
    target_prob = repeat(target_prob, k)  # Shape of [n_sentences * k]
    target_prob *= tokens_prob

    if n_sentences * k >= max_sentences:
        target, target_prob = topk_prob(target, target_prob, k=max_sentences)
    else:
        n_sentences *= k

    current_target_tokens += 1
```

In [18]:
def beautify(sentence: str) -> str:
    """Removes useless spaces.
    """
    punc = {'.', ',', ';'}
    for p in punc:
        sentence = sentence.replace(f' {p}', p)

    links = {'-', "'"}
    for l in links:
        sentence = sentence.replace(f'{l} ', l)
        sentence = sentence.replace(f' {l}', l)

    return sentence

In [19]:
def append_beams(
        target: torch.FloatTensor,
        beams: torch.FloatTensor
    ) -> torch.FloatTensor:
    """Add the beam tokens to the current sentences.
    Duplicate the sentences so one token is added per beam per batch.

    Args
    ----
        target: Batch of unfinished sentences.
            Shape of [batch_size, n_tokens].
        beams: Batch of beams for each sentences.
            Shape of [batch_size, n_beams].

    Output
    ------
        target: Batch of sentences with one beam per sentence.
            Shape of [batch_size * n_beams, n_tokens+1].
    """
    batch_size, n_beams = beams.shape
    n_tokens = target.shape[1]

    target = einops.repeat(target, 'b t -> b c t', c=n_beams)  # [batch_size, n_beams, n_tokens]
    beams = beams.unsqueeze(dim=2)  # [batch_size, n_beams, 1]

    target = torch.cat((target, beams), dim=2)  # [batch_size, n_beams, n_tokens+1]
    target = target.view(batch_size*n_beams, n_tokens+1)  # [batch_size * n_beams, n_tokens+1]
    return target


def beam_search(
        model: nn.Module,
        source: str,
        src_vocab: Vocab,
        tgt_vocab: Vocab,
        src_tokenizer,
        device: str,
        beam_width: int,
        max_target: int,
        max_sentence_length: int,
    ) -> list:
    """Do a beam search to produce probable translations.

    Args
    ----
        model: The translation model. Assumes it produces linear score (before softmax).
        source: The sentence to translate.
        src_vocab: The source vocabulary.
        tgt_vocab: The target vocabulary.
        device: Device to which we make the inference.
        beam_width: Number of top-k tokens we keep at each stage.
        max_target: Maximum number of target sentences we keep at the end of each stage.
        max_sentence_length: Maximum number of tokens for the translated sentence.

    Output
    ------
        sentences: List of sentences orderer by their likelihood.
    """
    src_tokens = ['<bos>'] + src_tokenizer(source) + ['<eos>']
    src_tokens = src_vocab(src_tokens)

    tgt_tokens = ['<bos>']
    tgt_tokens = tgt_vocab(tgt_tokens)

    # To tensor and add unitary batch dimension
    src_tokens = torch.LongTensor(src_tokens).to(device)
    tgt_tokens = torch.LongTensor(tgt_tokens).unsqueeze(dim=0).to(device)
    target_probs = torch.FloatTensor([1]).to(device)
    model.to(device)

    EOS_IDX = tgt_vocab['<eos>']
    with torch.no_grad():
        while tgt_tokens.shape[1] < max_sentence_length:
            batch_size, n_tokens = tgt_tokens.shape

            # Get next beams
            src = einops.repeat(src_tokens, 't -> b t', b=tgt_tokens.shape[0])
            predicted = model.forward(src, tgt_tokens)
            predicted = torch.softmax(predicted, dim=-1)
            probs, predicted = predicted[:, -1].topk(k=beam_width, dim=-1)

            # Separe between terminated sentences and the others
            idx_terminated, idx_not_terminated = indices_terminated(tgt_tokens, EOS_IDX)
            idx_terminated, idx_not_terminated = idx_terminated.to(device), idx_not_terminated.to(device)

            tgt_terminated = torch.index_select(tgt_tokens, dim=0, index=idx_terminated)
            tgt_probs_terminated = torch.index_select(target_probs, dim=0, index=idx_terminated)

            filter_t = lambda t: torch.index_select(t, dim=0, index=idx_not_terminated)
            tgt_others = filter_t(tgt_tokens)
            tgt_probs_others = filter_t(target_probs)
            predicted = filter_t(predicted)
            probs = filter_t(probs)

            # Add the top tokens to the previous target sentences
            tgt_others = append_beams(tgt_others, predicted)

            # Add padding to terminated target
            padd = torch.zeros((len(tgt_terminated), 1), dtype=torch.long, device=device)
            tgt_terminated = torch.cat(
                (tgt_terminated, padd),
                dim=1
            )

            # Update each target sentence probabilities
            tgt_probs_others = torch.repeat_interleave(tgt_probs_others, beam_width)
            tgt_probs_others *= probs.flatten()
            tgt_probs_terminated *= 0.999  # Penalize short sequences overtime

            # Group up the terminated and the others
            target_probs = torch.cat(
                (tgt_probs_others, tgt_probs_terminated),
                dim=0
            )


            tgt_tokens = torch.cat(
                (tgt_others, tgt_terminated),
                dim=0
            )

            # Keep only the top `max_target` target sentences
            if target_probs.shape[0] <= max_target:
                continue

            target_probs, indices = target_probs.topk(k=max_target, dim=0)
            tgt_tokens = torch.index_select(tgt_tokens, dim=0, index=indices)

    sentences = []
    for tgt_sentence in tgt_tokens:
        tgt_sentence = tgt_sentence[1:].tolist()  # Remove <bos> token and convert to Python list
        tgt_sentence = list(takewhile(lambda t: t != EOS_IDX, tgt_sentence))
        tgt_sentence = ' '.join(tgt_vocab.lookup_tokens(tgt_sentence))
        sentences.append(tgt_sentence)

    sentences = [beautify(s) for s in sentences]

    # Join the sentences with their likelihood
    sentences = [(s, p.item()) for s, p in zip(sentences, target_probs)]
    # Sort the sentences by their likelihood
    sentences = [(s, p) for s, p in sorted(sentences, key=lambda k: k[1], reverse=True)]

    return sentences

# Training loop
This is a basic training loop code. It takes a big configuration dictionnary to avoid never ending arguments in the functions.
We use [Weights and Biases](https://wandb.ai/) to log the trainings.
It logs every training informations and model performances in the cloud.
You have to create an account to use it. Every accounts are free for individuals or research teams.

In [20]:
def print_logs(dataset_type: str, logs: dict):
    """Print the logs.

    Args
    ----
        dataset_type: Either "Train", "Eval", "Test" type.
        logs: Containing the metric's name and value.
    """
    desc = [
        f'{name}: {value:.2f}'
        for name, value in logs.items()
    ]
    desc = '\t'.join(desc)
    desc = f'{dataset_type} -\t' + desc
    desc = desc.expandtabs(5)
    print(desc)


def topk_accuracy(
        real_tokens: torch.FloatTensor,
        probs_tokens: torch.FloatTensor,
        k: int,
        tgt_pad_idx: int,
    ) -> torch.FloatTensor:
    """Compute the top-k accuracy.
    We ignore the PAD tokens.

    Args
    ----
        real_tokens: Real tokens of the target sentence.
            Shape of [batch_size * n_tokens].
        probs_tokens: Tokens probability predicted by the model.
            Shape of [batch_size * n_tokens, n_target_vocabulary].
        k: Top-k accuracy threshold.
        src_pad_idx: Source padding index value.

    Output
    ------
        acc: Scalar top-k accuracy value.
    """
    total = (real_tokens != tgt_pad_idx).sum()

    _, pred_tokens = probs_tokens.topk(k=k, dim=-1)  # [batch_size * n_tokens, k]
    real_tokens = einops.repeat(real_tokens, 'b -> b k', k=k)  # [batch_size * n_tokens, k]

    good = (pred_tokens == real_tokens) & (real_tokens != tgt_pad_idx)
    acc = good.sum() / total
    return acc


def loss_batch(
        model: nn.Module,
        source: torch.LongTensor,
        target: torch.LongTensor,
        config: dict,
    )-> dict:
    """Compute the metrics associated with this batch.
    The metrics are:
        - loss
        - top-1 accuracy
        - top-5 accuracy
        - top-10 accuracy

    Args
    ----
        model: The model to train.
        source: Batch of source tokens.
            Shape of [batch_size, n_src_tokens].
        target: Batch of target tokens.
            Shape of [batch_size, n_tgt_tokens].
        config: Additional parameters.

    Output
    ------
        metrics: Dictionnary containing evaluated metrics on this batch.
    """
    device = config['device']
    loss_fn = config['loss'].to(device)
    metrics = dict()

    source, target = source.to(device), target.to(device)
    target_in, target_out = target[:, :-1], target[:, 1:]

    # Loss
    pred = model(source, target_in)  # [batch_size, n_tgt_tokens-1, n_vocab]
    pred = pred.view(-1, pred.shape[2])  # [batch_size * (n_tgt_tokens - 1), n_vocab]
    target_out = target_out.flatten()  # [batch_size * (n_tgt_tokens - 1),]
    metrics['loss'] = loss_fn(pred, target_out)

    # Accuracy - we ignore the padding predictions
    for k in [1, 5, 10]:
        metrics[f'top-{k}'] = topk_accuracy(target_out, pred, k, config['tgt_pad_idx'])

    return metrics


def eval_model(model: nn.Module, dataloader: DataLoader, config: dict) -> dict:
    """Evaluate the model on the given dataloader.
    """
    device = config['device']
    logs = defaultdict(list)

    model.to(device)
    model.eval()

    with torch.no_grad():
        for source, target in dataloader:
            metrics = loss_batch(model, source, target, config)
            for name, value in metrics.items():
                logs[name].append(value.cpu().item())

    for name, values in logs.items():
        logs[name] = np.mean(values)
    return logs


def train_model(model: nn.Module, config: dict):
    """Train the model in a teacher forcing manner.
    """
    train_loader, val_loader = config['train_loader'], config['val_loader']
    train_dataset, val_dataset = train_loader.dataset.dataset, val_loader.dataset.dataset
    optimizer = config['optimizer']
    clip = config['clip']
    device = config['device']

    columns = ['epoch']
    for mode in ['train', 'validation']:
        columns += [
            f'{mode} - {colname}'
            for colname in ['source', 'target', 'predicted', 'likelihood']
        ]
    log_table = wandb.Table(columns=columns)


    print(f'Starting training for {config["epochs"]} epochs, using {device}.')
    for e in range(config['epochs']):
        print(f'\nEpoch {e+1}')

        model.to(device)
        model.train()
        logs = defaultdict(list)

        for batch_id, (source, target) in enumerate(train_loader):
            optimizer.zero_grad()

            metrics = loss_batch(model, source, target, config)
            loss = metrics['loss']

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

            for name, value in metrics.items():
                logs[name].append(value.cpu().item())  # Don't forget the '.item' to free the cuda memory

            if batch_id % config['log_every'] == 0:
                for name, value in logs.items():
                    logs[name] = np.mean(value)

                train_logs = {
                    f'Train - {m}': v
                    for m, v in logs.items()
                }
                wandb.log(train_logs)
                logs = defaultdict(list)

        # Logs
        if len(logs) != 0:
            for name, value in logs.items():
                logs[name] = np.mean(value)
            train_logs = {
                f'Train - {m}': v
                for m, v in logs.items()
            }
        else:
            logs = {
                m.split(' - ')[1]: v
                for m, v in train_logs.items()
            }

        print_logs('Train', logs)

        logs = eval_model(model, val_loader, config)
        print_logs('Eval', logs)
        val_logs = {
            f'Validation - {m}': v
            for m, v in logs.items()
        }

        val_source, val_target = val_dataset[ torch.randint(len(val_dataset), (1,)) ]
        val_pred, val_prob = beam_search(
            model,
            val_source,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            device,  # It can take a lot of VRAM
            beam_width=10,
            max_target=100,
            max_sentence_length=config['max_sequence_length'],
        )[0]
        print(val_source)
        print(val_pred)

        logs = {**train_logs, **val_logs}  # Merge dictionnaries
        wandb.log(logs)  # Upload to the WandB cloud

        # Table logs
        train_source, train_target = train_dataset[ torch.randint(len(train_dataset), (1,)) ]
        train_pred, train_prob = beam_search(
            model,
            train_source,
            config['src_vocab'],
            config['tgt_vocab'],
            config['src_tokenizer'],
            device,  # It can take a lot of VRAM
            beam_width=10,
            max_target=100,
            max_sentence_length=config['max_sequence_length'],
        )[0]

        data = [
            e + 1,
            train_source, train_target, train_pred, train_prob,
            val_source, val_target, val_pred, val_prob,
        ]
        log_table.add_data(*data)

    # Log the table at the end of the training
    wandb.log({'Model predictions': log_table})

# Training the models
We can now finally train the models.
Choose the right hyperparameters, play with them and try to find
ones that lead to good models and good training curves.
Try to reach a loss under 1.0.

So you know, it is possible to get descent results with approximately 20 epochs.
With CUDA enabled, one epoch, even on a big model with a big dataset, shouldn't last more than 10 minutes.
A normal epoch is between 1 to 5 minutes.

*This is considering Colab Pro, we should try using free Colab to get better estimations.*

---

To test your implementations, it is easier to try your models
in a CPU instance. Indeed, Colab reduces your GPU instances priority
with the time you recently past using GPU instances. It would be
sad to consume all your GPU time on implementation testing.
Moreover, you should try your models on small datasets and with a small number of parameters.
For exemple, you could set:
```
MAX_SEQ_LEN = 10
MIN_TOK_FREQ = 20
dim_embedding = 40
dim_hidden = 60
n_layers = 1
```

You usually don't want to log anything onto WandB when testing your implementation.
To deactivate WandB without having to change any line of code, you can type `!wandb offline` in a cell.

Once you have rightly implemented the models, you can train bigger models on bigger datasets.
When you do this, do not forget to change the runtime as GPU (and use `!wandb online`)!

In [21]:
# Checking GPU and logging to wandb (if desired)

# !wandb login

!nvidia-smi

Wed Apr 22 23:09:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   30C    P0             52W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [22]:
# Instanciate the datasets

MAX_SEQ_LEN = 60
MIN_TOK_FREQ = 2
train_dataset, val_dataset = build_datasets(
    MAX_SEQ_LEN,
    MIN_TOK_FREQ,
    en_tokenizer,
    fr_tokenizer,
    train,
    valid,
)


print(f'English vocabulary size: {len(train_dataset.en_vocab):,}')
print(f'French vocabulary size: {len(train_dataset.fr_vocab):,}')

print(f'\nTraining examples: {len(train_dataset):,}')
print(f'Validation examples: {len(val_dataset):,}')

English vocabulary size: 7,523
French vocabulary size: 10,766

Training examples: 71,434
Validation examples: 24,053


In [23]:
# Print vocabulary information
print("\n" + "="*60)
print("VOCABULARY INFORMATION")
print("="*60)

# Print vocabulary sizes
print(f"\nEnglish vocabulary size: {len(train_dataset.en_vocab):,}")
print(f"French vocabulary size: {len(train_dataset.fr_vocab):,}")

# Print special tokens
print("\nSpecial tokens:")
for special in ['<unk>', '<pad>', '<bos>', '<eos>']:
    en_idx = train_dataset.en_vocab[special]
    fr_idx = train_dataset.fr_vocab[special]
    print(f"  {special}: EN={en_idx}, FR={fr_idx}")

# Print first 20 tokens from each vocabulary
print("\nFirst 20 English tokens:")
for i in range(min(20, len(train_dataset.en_vocab))):
    token = train_dataset.en_vocab.lookup_token(i)
    print(f"  {i}: {token}")

print("\nFirst 20 French tokens:")
for i in range(min(20, len(train_dataset.fr_vocab))):
    token = train_dataset.fr_vocab.lookup_token(i)
    print(f"  {i}: {token}")


VOCABULARY INFORMATION

English vocabulary size: 7,523
French vocabulary size: 10,766

Special tokens:
  <unk>: EN=0, FR=0
  <pad>: EN=1, FR=1
  <bos>: EN=2, FR=2
  <eos>: EN=3, FR=3

First 20 English tokens:
  0: <unk>
  1: <pad>
  2: <bos>
  3: <eos>
  4: What
  5: an
  6: opportunity
  7: !
  8: Our
  9: sales
  10: are
  11: decreasing
  12: .
  13: You
  14: should
  15: rest
  16: a
  17: little
  18: Someone
  19: stole

First 20 French tokens:
  0: <unk>
  1: <pad>
  2: <bos>
  3: <eos>
  4: Quelle
  5: occasion
  6: !
  7: Nos
  8: ventes
  9: .
  10: Tu
  11: devrais
  12: te
  13: reposer
  14: un
  15: peu
  16: Quelqu'
  17: a
  18: volé
  19: mon


# Your Question Responses
Please see at the top of the Jupyter notebook

#BLEU score function

In [24]:
!pip install sacrebleu -q

import sacrebleu
import time

def compute_bleu(model, dataset, config, n_samples=200):
    """Compute BLEU score on n_samples from the dataset."""
    model.eval()
    hypotheses = []
    references = []

    indices = torch.randperm(len(dataset))[:n_samples]

    for idx in indices:
        en_sentence, fr_sentence = dataset.dataset[idx]
        try:
            pred, _ = beam_search(
                model,
                en_sentence,
                config['src_vocab'],
                config['tgt_vocab'],
                config['src_tokenizer'],
                config['device'],
                beam_width=5,
                max_target=50,
                max_sentence_length=config['max_sequence_length']
            )[0]
            hypotheses.append(pred)
            references.append(fr_sentence)
        except:
            continue

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00


#Training time + BLEU summary table

In [25]:
# I am using here the overall run summary scores to evaluate which is from the data at the 20th epoch,
# im using the validation top 1 data to see how well the model generalizes to unseen data

results_summary = {
    'RNN':             {'train_loss': 1.82864, 'val_loss': 2.20972, 'top1': 0.54895},
    'GRU':             {'train_loss': 1.23275, 'val_loss': 1.6665, 'top1': 0.64338},
    'Transformer(4 heads)': {'train_loss': 0.82834, 'val_loss': 1.29244, 'top1': 0.71651},
    'Transformer(14 heads)':{'train_loss': 0.68695, 'val_loss': 1.24136, 'top1': 0.73522},
}
"""
print("Computing BLEU score...")
start = time.time()
bleu = compute_bleu(model, val_dataset, config, n_samples=200)
elapsed = time.time() - start
print(f"BLEU Score: {bleu:.2f}")
print(f"(Computed in {elapsed:.0f}s)")
"""

'\nprint("Computing BLEU score...")\nstart = time.time()\nbleu = compute_bleu(model, val_dataset, config, n_samples=200)\nelapsed = time.time() - start\nprint(f"BLEU Score: {bleu:.2f}")\nprint(f"(Computed in {elapsed:.0f}s)")\n'

# Discussion des expériences

Dans cette section, on va mettre en évidence les résultats obtenus lors des différentes expériences menées dans ce TP. L'objectif est de comparer trois architectures de traduction automatique (RNN, GRU, Transformer), d'analyser l'effet de plusieurs hyperparamètres et d'évaluer différentes stratégies de décodage. Toutes les expériences ont été réalisées sur 33 % du dataset anglais-français, avec un maximum de 20 époques d'entraînement.

## 1. Comparaison des architectures : RNN vs GRU vs Transformer

### Résultats obtenus

| Modèle               | Perte entraînement | Perte validation | Précision Top-1 |
|----------------------|--------------------|-----------------|-----------------|
| RNN                    | 1.82864  | 2.20972 | 0.54895 |
| GRU                    | 1.23275  | 1.6665  | 0.64338 |
| Transformer (4 heads ) | 0.82834  | 1.29244 | 0.71651 |
| Transformer (14 heads) | 0.68605  | 1.24136 | 0.73522 |

Tout d'abord, on peut constater que pour le RNN, on obtient les performances les plus faibles, avec une précision Top-1 de seulement 0.54895. Cela confirme le fait que le RNN souffre du problème de disparition du gradient sur les longues séquences, ce qui l'empêche de mémoriser le début d'une phrase lorsqu'il génère la fin de la traduction. L'écart important entre sa perte d'entraînement (1.82864) et celle de validation (2.20972) indique également qu'il a du mal à généraliser.

Pour le GRU, on voit clairement une amélioration au niveau des résultats par rapport au RNN. On peut donc supposer que les portes de mise à jour et de réinitialisation lui permettent de mieux retenir les informations importantes sur de plus longues distances, ce qui se traduit directement par de meilleures traductions. L'écart entraînement/validation reste présent mais se réduit, ce qui montre qu'on a possiblement une meilleure généralisation que le RNN.

Enfin, pour le Transformer, sa performance surpasse largement les deux modèles récurrents, atteignant 0.71651 de précision avec seulement 4 têtes d'attention, ce qui met en évidence l'avantage du mécanisme d'auto-attention où chaque mot peut directement "regarder" tous les autres mots de la phrase en une seule opération, sans être limité par une mémoire séquentielle qui s'estompe. De plus, le traitement parallèle accélère considérablement l'entraînement.

Bref, il est important de noter que malgré sa performance, qui est nettement supérieure aux autres modèles, le Transformer présente aussi un écart entraînement/validation (0.82834 vs 1.29244), ce qui signifie qu'il y a encore une légère tendance au surapprentissage, probablement amplifiée par la taille réduite du dataset utilisé (33%).

### 2. Effet de la taille du dataset utilisée sur le Transformer (4 têtes)

| Dataset | Perte entraînement | Perte validation | Précision Top-1 | Écart pertes (par différence) |
|---------|--------------------|-----------------|-----------------|-----------------|
|  5%  |  0.63692   |  2.18732   |  0.58317  |  1.55040  |
|  33% |  0.82834   |  1.29244   |  0.71651  |  0.46410  |
|  75% |  0.95521   |  1.03919   |  0.75700  |  0.08398 |
|  90% |  0.95939   |  1.00967   |  0.76282  |  0.05028  |

À partir de ces résultats, on peut faire plusieurs observations. Premièrement, on peut constater que la perte d'entraînement augmente au fur et à mesure qu'on prend un plus grand nombre de données, ce qui peut être contre-intuitif à première vue, car on aurait pu s'attendre à ce qu'un modèle entraîné sur plus de données ait une perte d'entraînement plus basse. Il se peut donc que les modèles entraînés sur peu de données surapprennent. Ainsi, avec seulement 5 % ou 33 % des données, le modèle finissait par mémoriser les exemples d'entraînement, ce qui créait une perte d'entraînement artificiellement basse (0.63692 à 5 % du dataset et 0.82834 à 33 % du dataset). Alors qu'avec 75 % et 90 % des données, le modèle ne peut plus mémoriser et il doit donc désormais vraiment apprendre des patterns généraux, ce qui donne une perte d'entraînement légèrement plus haute (0.95939 pour 90 % du dataset par exemple), mais qui représente une meilleure généralisation.

Dans un deuxième temps, on peut constater que l'écart entre les pertes d'entraînement et de validation diminue à mesure qu'on s'entraîne avec plus de données. En effet, à 5 % du dataset, l'écart est énorme (1.55040). À 33 % du dataset, cet écart se réduit déjà à 0.46410, puis à 0.08398 à 75 % et finalement à seulement 0.05028 à 90 %. Cela confirme que le surapprentissage observé était principalement dû à la taille réduite du dataset utilisé, et non à l'architecture elle-même. On peut donc en conclure qu'avec suffisamment de données, le Transformer généralise très bien.

Dans un troisième temps, on peut constater que la précision Top-1 n'est pas uniforme à travers les différentes tailles de données. En effet, on peut voir que cette précision augmente au fur et à mesure qu'on augmente la taille du dataset.

Enfin, on peut clairement voir que la courbe de performance s'aplatit au fur et à mesure que la taille du dataset utilisé augmente. En effet, on observe un plus grand gain de performance lorsqu'on passe de 5 % du dataset à 33 % du dataset, et que ce gain diminue progressivement au fur et à mesure qu'on augmente la taille du dataset utilisé. On peut donc supposer qu'on s'approche d'un seuil limite pour cette architecture et ces hyperparamètres. Ainsi, pour progresser davantage, il faudrait probablement augmenter la capacité du modèle (par exemple : plus de têtes, plus de couches ou une plus grande dimension d_model).

Bref, ces résultats montrent que la taille du dataset est un facteur qui peut apporter certaines limites aux expériences. En effet, un Transformer avec seulement 4 têtes entraîné sur 5 % des données est peut-être largement inférieur au même modèle mais entraîné sur 90 % du dataset, sans aucun changement d'architecture.


## 3. Effet du nombre de têtes d'attention pour le Transformer (4 vs 14 heads)

### Résultats obtenus

| Modèle                | Perte entraînement | Perte validation | Précision Top-1 |
|-----------------------|--------------------|-----------------|-----------------|
| Transformer (4 têtes)  | 0.82834  | 1.29244 | 0.71651  |
| Transformer (14 têtes) | 0.68605  | 1.24136 | 0.73522  |


À partir de ces résultats, on peut constater que le fait d'augmenter le nombre de têtes de 4 à 14 apporte quand même une amélioration modeste pour la précision, mais significative pour les pertes.

À cet effet, on peut supposer qu'avec plus de têtes, le modèle peut apprendre davantage de types de relations en parallèle. En effet, comme certaines têtes se spécialisent sur la syntaxe, d'autres sur la sémantique, d'autres encore sur les relations à longue portée. Plus le nombre de perspectives est grand, plus il est raisonnable de penser que la représentation du contexte est riche.

Quant aux pertes, on peut déduire que chaque tête possède une dimension relativement réduite (dim_embedding / n_heads). Avec 14 têtes sur 196 dimensions, chaque tête ne dispose que d'environ 14 dimensions, ce qui peut limiter sa capacité. Il y aurait donc un certain compromis où le fait d'avoir plus de têtes offre plus de perspectives, mais chaque perspective est alors moins détaillée.

Bref, on peut en conclure que pour un modèle de taille fixe, un nombre de têtes intermédiaire (4 à 8) est sûrement suffisant. Ainsi, le fait d'augmenter davantage le nombre de têtes n'apporte pas nécessairement des pertes plus faibles.



## 4. Effet du nombre de couches (p.ex: 1, 3 et 5 couches)

Cette expérience a pour but de tester comment la profondeur du modèle influence les performances des modèles.

| Modèle                | Perte entraînement | Perte validation | Précision Top-1 |
|-----------------------|--------------------|-----------------|-----------------|
| 1 couche  | 1.33953  | 1.60478 | 0.65012  |
| 3 couches | 0.8394   | 1.31299 | 0.71651  |
| 5 couches | 0.70445  | 1.25187 | 0.72588  |

À partir de ces résultats, on peut déduire que plus le modèle est profond, meilleures sont les performances, aussi bien en perte qu'en précision.

Avec le modèle à 1 couche, on obtient la perte d'entraînement la plus haute (1.33953) et la précision la plus faible (0.65012). Cela confirme qu'une seule couche d'attention n'est pas suffisante pour capturer toute la complexité des relations entre les mots d'une phrase, ce qui est cohérent puisque le modèle est trop contraint. En effet, il n'a qu'une seule chance de "regarder" le contexte avant de produire une représentation, ce qui limite fortement sa capacité à traduire des structures grammaticales plus complexes.

Quand on passe à 3 couches, on observe de meilleures performances. En effet, pour la perte d'entraînement, on passe de 1.33953 à 0.8394. La perte de validation descend de 1.60478 à 1.31299 et la précision monte de 0.65012 à 0.71651. On peut donc supposer que chaque couche supplémentaire permet au modèle de raffiner progressivement sa compréhension du contexte.

Enfin, lorsqu'on passe à 5 couches, on voit encore une amélioration, mais nettement plus modeste. En effet, pour la précision, on ne passe que de 0.71651 à 0.72588, une perte de validation qui passe de 1.31299 à 1.25187 et une perte d'entraînement qui passe de 0.8394 à 0.70445. Ainsi, on peut noter qu'au-delà d'un certain nombre de couches, les gains marginaux diminuent, car le modèle a déjà appris l'essentiel des relations utiles. De plus, avec seulement 33 % du dataset, un modèle plus profond risque davantage de surapprendre, comme le montre l'écart qui augmente entre la perte d'entraînement (0.70445) et la perte de validation (1.25187).

Bref, on peut conclure que 3 couches offrent le meilleur compromis capacité/généralisation sur ce dataset. Avec un dataset complet (100 %), on pourrait s'attendre à ce que 5 couches surpassent davantage 3 couches, car plus de données permettraient de mieux exploiter les avantages qui y viennent avec un modèle plus profond.

## 5. Effet du dropout (0.0, 0.1 et 0.3)

Dans cette expérience, on va utiliser la technique du dropout, qui est une technique de régularisation où, pendant l'entraînement, on éteint aléatoirement un certain pourcentage de neurones à chaque étape, ce qui force le modèle à ne pas trop dépendre de certains neurones spécifiques et à mieux généraliser.

| Modèle                | Perte entraînement | Perte validation | Précision Top-1 |
|-----------------------|--------------------|-----------------|-----------------|
| dropout = 0.0 | 0.71404  | 1.46285 | 0.68984  |
| dropout = 0.1 | 1.10369  | 1.33507 | 0.69735  |
| dropout = 0.3 | 1.71492  | 1.57634 | 0.6528  |

Pour un dropout = 0.0, on peut voir qu'on obtient la perte d'entraînement la plus basse (0.71404), ce qui est attendu étant donné que, sans aucune contrainte, le modèle apprend librement sur les données d'entraînement. Cependant, sa perte de validation (1.46285) est nettement plus haute que sa perte d'entraînement, ce qui peut indiquer qu'on a du surapprentissage. De ce fait, on peut aussi supposer que le modèle a mémorisé les données vues, mais généralise moins bien sur des phrases nouvelles. Sa précision Top-1 de 0.68984 est la deuxième meilleure des trois configurations, mais ce résultat reste peu pertinent, car il repose sur une mémorisation plutôt que sur une vraie compréhension du langage.

Quant à un dropout = 0.1, on constate que sa perte d'entraînement est plus haute (1.10369), ce qui est logique car le dropout perturbe l'apprentissage, mais en contrepartie, la perte de validation est la plus basse (1.33507) et la précision Top-1 est la meilleure (0.69735). L'écart entraînement/validation se réduit de beaucoup comparé à un dropout = 0.0, ce qui peut être un signe que le modèle a appris des patterns généraux plutôt que de mémoriser les exemples.

Enfin, pour un dropout = 0.3, on peut voir que cela donne les résultats les plus faibles dans l'ensemble. En effet, sa perte d'entraînement est la plus haute (1.71492), sa perte de validation est aussi la plus haute (1.57634), et sa précision chute à 0.6528. On peut donc déduire qu'en éteignant 30 % des neurones à chaque étape, le modèle est trop contraint pour apprendre correctement les patterns complexes de la traduction. Il n'arrive ni à bien s'ajuster aux données d'entraînement ni à bien généraliser, ce qui peut indiquer qu'on a un cas de sous-apprentissage.

Bref, les résultats montrent qu'avec un dropout = 0.1, c'est clairement optimal pour ce contexte. Un dropout trop faible (dropout = 0.0) peut mener au surapprentissage, et un dropout trop élevé (dropout = 0.3) peut empêcher le modèle d'apprendre suffisamment, ce qui dégrade les performances sur les deux ensembles.

### 6. Comparaison : Greedy Search vs Beam Search

Dans cette expérience, on va comparer greedy search et beam search. En principe, greedy search choisit à chaque étape le token ayant la probabilité la plus haute, sans jamais revenir en arrière. C'est simple et rapide, mais une mauvaise décision au début de la séquence ne peut jamais être corrigée. Quant à beam search (beam_width = 10), on explore 10 hypothèses en parallèle à chaque étape. Il garde les 10 séquences partielles les plus probables et les développe toutes, avant de retourner la meilleure séquence complète à la fin.

### Résultats obtenus

| Phrase anglaise              | Greedy                        | Beam Search                              |
|------------------------------|-------------------------------|------------------------------------------|
| "I love you."                | `<unk> <unk> <unk> <unk>` (0.3769484758377075)    | Je vous adore. (35.71%)                  |
| "What time is it?"           | `<unk> <unk> <unk> <unk> <unk> <unk>` (0.3171319365501404) | À quelle heure est-il ? (30.11%) |
| "She is a student."          | `<unk> <unk> <unk> <unk>` (0.5677785277366638)    | Elle est étudiante. (53.79%)             |
| "The weather is nice today." | `<unk> <unk> <unk> <unk> <unk> <unk> <unk>` (0.15324589610099792) | Le temps est un beau aujourd'hui. (14.56%) |
| "You can try your work here."| `<unk> <unk> <unk> <unk> <unk> <unk> <unk>` (0.6002342104911804) | Tu peux essayer ton travail ici. (15.35%) |


D'après ces résultats, on peut voir que greedy search produit systématiquement des tokens <unk> (token inconnu) pour toutes les phrases qu'on a testées, ce qui donne des traductions complètement inutilisables. Cela peut s'expliquer par le fait que dès que le premier token prédit est <unk>, le décodeur se retrouve dans un état incohérent qui le pousse à continuer de prédire <unk>, car c'est le token qui maximise localement la probabilité à chaque étape suivante. On a donc une propagation d'erreur où une seule mauvaise décision initiale peut corrompre toute la séquence, ce qui est un inconvénient de greedy search.

Quant à beam search, en revanche, il produit des traductions relativement cohérentes et correctes sur la majorité des phrases. Par exemple, pour la phrase "She is a student.", on obtient "Elle est étudiante." avec 53.79 % de confiance, ou encore pour la phrase "You can try your work here.", on obtient "Tu peux essayer ton travail ici." (15.35 %), qui sont des traductions pas totalement précises, mais dont on comprend le sens. On peut donc supposer que le fait d'explorer 10 hypothèses simultanément lui permet d'éviter le problème qui vient du <unk> initial et de trouver un chemin valide vers une traduction complète, même si elle n'est pas parfaitement exacte.

Toutefois, on observe que le beam search n'est pas parfait non plus. En effet, pour d'autres exemples de phrases comme "The weather is nice today.", on obtient "Le temps est un beau aujourd'hui.". On voit que la structure grammaticale est un peu maladroite ("un beau" au lieu de "beau"). Cela montre que les limites viennent en fait du modèle lui-même, et non de la stratégie de décodage. En effet, le dataset utilisé est relativement petit (33 %) et n'a probablement pas permis au modèle d'apprendre toutes les structures grammaticales du français.

Le beam search propose également plusieurs alternatives classées par
probabilité. Par exemple pour la phrase "You can try your work here.", on a:
- 0. (15.35286%) 	 Tu peux essayer ton travail ici.
- 1. (6.38615%) 	 Vous pouvez essayer votre travail ici.
- 2. (5.78331%) 	 Tu peux essayer ton travail, ici.
- 3. (3.61915%) 	 Vous pouvez essayer votre travail, ici.
- 4. (1.91467%) 	 Vous pouvez essayer votre travail ici ?

Bref, cette liste de traductions alternatives montre que le modèle est capable de distinguer correctement le tutoiement ("tu/ton") du vouvoiement ("vous/votre"), deux traductions qui peuvent être valides selon le contexte. Bref, greedy search est inutilisable d'après les résultats obtenus, alors que beam search donne des résultats qui peuvent être acceptables.

## 7. Score BLEU : mesure pour la qualité de traduction

Cette expérience a pour but d'explorer le score BLEU, qui est la métrique de référence pour la traduction automatique. Il mesure à quel point la traduction générée par le modèle ressemble à une traduction humaine, en comparant des séquences de mots consécutifs (n-grammes). Contrairement à la perte (cross-entropie) et à la précision Top-1, qui mesurent la qualité token par token, le BLEU évalue la cohérence de la phrase entière, ce qui est bien plus représentatif de la qualité réelle d'une traduction. En ce qui concerne les scores ci-dessous, ils ont été calculés avec beam search (beam_width = 5) sur 200 échantillons aléatoires de l'ensemble de validation.


### Scores BLEU du transformer

| Modèle                  | Score BLEU | Temps de calcul |
|-------------------------|------------|-----------------|
| Transformer (14 heads)  | **37.21**  | 176 sec.        |

Pour le Transformer avec 14 têtes, il atteint un score BLEU de 37.21, ce qui est considéré comme une qualité de traduction relativement bonne, proche du niveau humain. C'est un résultat quand même solide en tenant compte du fait qu'on n'ait utilisé que 33 % du dataset. Malheureusement, les scores BLEU du RNN, du GRU et du Transformer à 4 têtes n'ont pas été calculés séparément. Seules leurs métriques de perte et Top-1 sont disponibles dans le tableau de comparaison de la section 1.


### Scores BLEU — Effet du nombre de couches du transformer

| Modèle          | Score BLEU | Temps d'entraînement |
|-----------------|------------|----------------------|
| 1 couche        | 24.65      | 24.1 min             |
| 3 couches       | **34.35**  | 58.4 min             |
| 5 couches       | 31.82      | 92.4 min             |

En ce qui concerne l'effet du nombre de couches, les résultats permettent de faire quelques constats par rapport à ce qu'on observait avec la précision Top-1. Avec Top-1, 5 couches semblaient légèrement meilleures que 3 couches (0.72588 vs 0.71651), mais selon le score BLEU, ce n'est pas le cas. En effet, comme on peut le voir dans le tableau ci-dessus, à 3 couches, on obtient un meilleur score BLEU (34.35) que 5 couches (31.82). Cela confirme que 5 couches surapprenait les données d'entraînement : il prédisait bien les tokens individuels, mais produisait des phrases moins cohérentes dans l'ensemble. De plus, le temps d'entraînement de 5 couches (92.4 min) est presque le double de celui à 3 couches (58.4 min) pour un résultat final inférieur, ce qui rend 3 couches clairement le meilleur compromis performance/coût.

### Scores BLEU — Effet du dropout

| Modèle        | Score BLEU | Époques |
|---------------|------------|---------|
| dropout = 0.0 | 29.10      | 10      |
| dropout = 0.1 | **29.21**  | 10      |
| dropout = 0.3 | 24.63      | 10      |

À partir de ces résultats, on peut voir que les scores BLEU confirment les tendances observées en Top-1 (accuracy). En effet, un dropout = 0.1 permet d'obtenir le meilleur score (29.21), un score légèrement supérieur à dropout = 0.0 (29.10). L'écart est relativement petit, mais cohérent avec l'idée que la régularisation aide à produire des traductions plus généralisables. Quant à un dropout = 0.3, on peut voir qu'on décroche clairement avec un score BLEU de 24.63, ce qui correspond à une traduction acceptable, ce qui soutient l'idée de sous-apprentissage.
Par ailleurs, il est important de noter que ces expériences avec dropout n'ont été entraînées que sur 10 époques (contre 20 pour les autres auparavant), ce qui peut légèrement défavoriser les scores BLEU par rapport aux autres expériences.

# **Project Section**

In [26]:
# Testing gpu in use...
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


## **Configs pour les expériences Mamba**

In [27]:
# Config main pour la traduction
MAMBA_TRANSLATION_BASE = {
    'dim_embedding': 256,
    'n_layers': 4,
    'd_state': 16,
    'd_conv': 4,
    'expand': 2,
    'dropout': 0.1,

    'epochs': 20,
    'batch_size': 256,
    'lr': 5e-4,
    'betas': (0.9, 0.98),
    'weight_decay': 1e-5,
    'clip': 1.0,
    'log_every': 100,
    'max_sequence_length': 40,
}

# Config plus rapide pour les ablations
MAMBA_TRANSLATION_FAST = {
    **MAMBA_TRANSLATION_BASE,
    'dim_embedding': 128,
    'n_layers': 3,
    'epochs': 10,
    'batch_size': 256,
}

# Config pour les tâches synthétiques
MAMBA_SYNTHETIC_BASE = {
    'vocab_size': 16,
    'd_model': 64,
    'n_layers': 2,
    'd_state': 16,
    'd_conv': 4,
    'expand': 2,
    'dropout': 0.0,
    'seq_len_train': 256,
    'n_to_copy': 16,
    'num_samples_train': 5000,
    'num_samples_val': 500,
    'num_steps': 3000,
    'batch_size': 32,
    'lr': 1e-3,
}

# Valeurs des sweeps
SWEEPS = {
    'd_state':  [4, 8, 16, 32],
    'n_layers': [2, 4, 6],
    'expand':   [1, 2, 4],
    'dropout':  [0.0, 0.1, 0.3],
    'A_init':   ['s4d_real', 's4d_complex', 'random'],
}

## **Setup & Class definitions**


In [28]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat


import numpy as np
from torch.utils.data import Dataset, DataLoader


from einops import repeat, rearrange


import time
import gc
import os


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.environ['WANDB_MODE'] = 'disabled'


### Normalisation RMS (remplace LayerNorm dans Mamba)

In [29]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization.

    Used in Mamba instead of LayerNorm (no bias term, slightly faster).

    Parameters
    ----------
        d_model: Dimension of the input features.
        eps: Small value for numerical stability.
    """
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()

        self.weight = nn.Parameter(torch.ones(d_model).to(device))
        self.eps = eps

    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        """
        Args
        ----
            x: Input tensor.
                Shape of [batch_size, seq_len, d_model].

        Output
        ------
            out: Normalized tensor.
                Shape of [batch_size, seq_len, d_model].
        """
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

### Scan selectif S6 in Mamba paper

In [30]:
class SelectiveSSM(nn.Module):
    """Selective State Space Model (the S6 layer of Mamba).

    The key difference from S4 is that the parameters Delta, B, C are
    functions of the input x (selective), allowing the model to
    selectively remember or forget information along the sequence.

    Parameters
    ----------
        d_inner: SSM input dimension (= E * d_model).
        d_state: State dimension N.
        dt_rank: Rank of the low-rank Delta projection.
        dt_min, dt_max: Init range for the Delta bias.
    """
    def __init__(
            self,
            d_inner: int,
            d_state: int = 16,
            dt_rank: int = None,
            dt_min: float = 0.001,
            dt_max: float = 0.1,
        ):
        super().__init__()

        self.d_inner = d_inner
        self.d_state = d_state
        if dt_rank is None:
            dt_rank = max(1, d_inner // 32)
        self.dt_rank = dt_rank

        # Single Linear that produces (Delta_raw, B, C) from x
        self.x_proj = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False).to(device)

        # Projection that broadcasts Delta from dt_rank up to d_inner
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True).to(device)

        # Special init of dt_proj bias so softplus(bias) falls in [dt_min, dt_max]
        dt_init_std = dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(
            torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

        # A stored as A_log so that A = -exp(A_log) stays negative (stability).
        # Init to S4D-Real: A_n = -(n+1).
        A = repeat(
            torch.arange(1, d_state + 1, dtype=torch.float),
            'n -> d n', d=d_inner,
        )
        self.A_log = nn.Parameter(torch.log(A).to(device))

        # D: skip-connection weight
        self.D = nn.Parameter(torch.ones(d_inner).to(device))


    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        """Run the selective scan on input x.

        Args
        ----
            x: Input sequence.
                Shape of [batch_size, seq_len, d_inner].

        Output
        ------
            y: Output sequence.
                Shape of [batch_size, seq_len, d_inner].
        """
        batch, sl, _ = x.shape
        A = -torch.exp(self.A_log)

        # Get (Delta_raw, B, C) from x
        x_dbl = self.x_proj(x)
        delta_raw, B, C = torch.split(
            x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1,
        )
        delta = F.softplus(self.dt_proj(delta_raw))

        # Discretize A and build input-to-state term
        delta_A = torch.exp(torch.einsum('bld,dn->bldn', delta, A))
        delta_B_x = torch.einsum('bld,bln,bld->bldn', delta, B, x)

        # Sequential scan
        h = torch.zeros(batch, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(sl):
            h = delta_A[:, t] * h + delta_B_x[:, t]
            y_t = torch.einsum('bdn,bn->bd', h, C[:, t])
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        y = y + x * self.D
        return y

### MambaBlock

In [31]:
class MambaBlock(nn.Module):
    """Full Mamba block (Figure 3 of the paper).

    Pipeline: in_proj -> (x, z); x through Conv1d -> SiLU -> SelectiveSSM;
    gate with SiLU(z); out_proj.

    Parameters
    ----------
        d_model: Model hidden dimension.
        d_state: SSM state size N.
        d_conv: Depthwise causal conv kernel size.
        expand: Expansion factor E for d_inner = E * d_model.
    """
    def __init__(
            self,
            d_model: int,
            d_state: int = 16,
            d_conv: int = 4,
            expand: int = 2,
        ):
        super().__init__()

        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_conv = d_conv

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False).to(device)

        # Depthwise causal conv. We pad on both sides then trim the right
        # side in forward() to keep causality.
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
            bias=True,
        ).to(device)

        self.ssm = SelectiveSSM(d_inner=self.d_inner, d_state=d_state)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False).to(device)


    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        """
        Args
        ----
            x: Input.
                Shape of [batch_size, seq_len, d_model].

        Output
        ------
            out: Block output.
                Shape of [batch_size, seq_len, d_model].
        """
        sl = x.size(1)

        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        # Depthwise causal conv. Conv1d needs [B, C, L] so we transpose.
        # After conv the length is L+d_conv-1, trim to L to stay causal.
        x_conv = rearrange(x_ssm, 'b l d -> b d l')
        x_conv = self.conv1d(x_conv)[:, :, :sl]
        x_conv = rearrange(x_conv, 'b d l -> b l d')
        x_conv = F.silu(x_conv)

        y = self.ssm(x_conv)
        y = y * F.silu(z)

        out = self.out_proj(y)
        return out

### MambaStack

In [32]:
class MambaStack(nn.Module):
    """Stack of Mamba blocks with pre-RMSNorm and residual connections.

    Each layer: x = x + Block(RMSNorm(x)).

    Parameters
    ----------
        d_model: Model hidden dimension.
        n_layers: Number of Mamba blocks.
        d_state, d_conv, expand: Passed to MambaBlock.
        dropout: Dropout rate.
    """
    def __init__(
            self,
            d_model: int,
            n_layers: int,
            d_state: int = 16,
            d_conv: int = 4,
            expand: int = 2,
            dropout: float = 0.1,
        ):
        super().__init__()

        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        """
        Args
        ----
            x: Input embeddings.
                Shape of [batch_size, seq_len, d_model].

        Output
        ------
            out: Stack output.
                Shape of [batch_size, seq_len, d_model].
        """
        for block, norm in zip(self.layers, self.norms):
            x = x + self.dropout(block(norm(x)))
        x = self.final_norm(x)
        return x

### Translation Mamba

In [33]:
class TranslationMamba(nn.Module):
    """Mamba encoder-decoder for a translation task.

    Uses Mamba as a decoder-only model with prefix conditioning: the
    source sentence is prepended to the target, and the causal recurrence
    lets source information flow into target-side predictions.

    Parameters
    ----------
        n_tokens_src: Number of tokens in the source vocabulary.
        n_tokens_tgt: Number of tokens in the target vocabulary.
        dim_embedding: Token embedding dimension.
        n_layers: Number of Mamba blocks.
        d_state: SSM state dimension.
        d_conv: Causal conv kernel size.
        expand: Expansion factor E.
        dropout: Dropout rate.
        src_pad_idx: Source padding index value.
        tgt_pad_idx: Target padding index value.
    """
    def __init__(
            self,
            n_tokens_src: int,
            n_tokens_tgt: int,
            dim_embedding: int,
            n_layers: int,
            d_state: int = 16,
            d_conv: int = 4,
            expand: int = 2,
            dropout: float = 0.1,
            src_pad_idx: int = 0,
            tgt_pad_idx: int = 0,
        ):
        super().__init__()

        self.dim_embedding = dim_embedding
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        self.embedding_src = nn.Embedding(n_tokens_src, dim_embedding, padding_idx=src_pad_idx).to(device)
        self.embedding_tgt = nn.Embedding(n_tokens_tgt, dim_embedding, padding_idx=tgt_pad_idx).to(device)
        self.emb_dropout = nn.Dropout(dropout)

        self.mamba = MambaStack(
            d_model=dim_embedding,
            n_layers=n_layers,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
            dropout=dropout,
        )
        self.out_layer = nn.Linear(dim_embedding, n_tokens_tgt).to(device)


    def forward(
            self,
            source: torch.LongTensor,
            target: torch.LongTensor,
        ) -> torch.FloatTensor:
        """Predict the next target token distribution at each position.

        Args
        ----
            source: Batch of source sentences.
                Shape of [batch_size, seq_len_src].
            target: Batch of target sentences.
                Shape of [batch_size, seq_len_tgt].

        Output
        ------
            y: Logits over the target vocabulary.
                Shape of [batch_size, seq_len_tgt, n_tokens_tgt].
        """
        src_emb = self.embedding_src(source) * math.sqrt(self.dim_embedding)
        tgt_emb = self.embedding_tgt(target) * math.sqrt(self.dim_embedding)
        src_emb = self.emb_dropout(src_emb)
        tgt_emb = self.emb_dropout(tgt_emb)

        # Concatenate source + target, run through the causal Mamba stack
        full = torch.cat([src_emb, tgt_emb], dim=1)
        out = self.mamba(full)

        # Keep only target-side positions for the output
        tgt_out = out[:, source.size(1):, :]

        y = self.out_layer(tgt_out)
        return y

## Helpers pour sauvegarder les résultats d'expériences

In [34]:
import json
import pickle
from pathlib import Path
import numpy as np

def save_results(results: dict, name: str, outdir: str = 'results'):
    """Sauvegarde un dict de résultats en pickle + JSON."""
    Path(outdir).mkdir(parents=True, exist_ok=True)

    pkl_path = Path(outdir) / f'{name}.pkl'
    with open(pkl_path, 'wb') as f:
        pickle.dump(results, f)
    print(f'Saved pickle: {pkl_path}')

    def _strip(obj):
        if isinstance(obj, dict):
            return {k: _strip(v) for k, v in obj.items()
                    if not isinstance(v, (torch.Tensor,))}
        if isinstance(obj, list):
            return [_strip(v) for v in obj]
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        if isinstance(obj, (torch.Tensor,)):
            return None
        return obj

    json_path = Path(outdir) / f'{name}.json'
    with open(json_path, 'w') as f:
        json.dump(_strip(results), f, indent=2, default=str)
    print(f'Saved JSON:   {json_path}')

In [35]:

def load_results(name: str, outdir: str = 'results') -> dict:
    with open(Path(outdir) / f'{name}.pkl', 'rb') as f:
        return pickle.load(f)

Training loop

In [36]:
print(type(train_dataset))
print(len(train_dataset))

<class '__main__.TranslationDataset'>
71434


In [37]:
# Configuration Mamba pour traduction
config_mamba = {
    'epochs': 20,
    'batch_size': 128,
    'lr': 5e-4,
    'betas': (0.9, 0.98),
    'weight_decay': 1e-5,
    'clip': 1.0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # Mamba-specific
    'dim_embedding': 256,
    'n_layers': 4,
    'd_state': 16,
    'd_conv': 4,
    'expand': 2,
    'dropout': 0.1,

    'n_tokens_src': len(train_dataset.en_vocab),
    'n_tokens_tgt': len(train_dataset.fr_vocab),
    'src_vocab': train_dataset.en_vocab,
    'tgt_vocab': train_dataset.fr_vocab,
    'src_tokenizer': en_tokenizer,
    'tgt_tokenizer': fr_tokenizer,
    'src_pad_idx': train_dataset.en_vocab['<pad>'],
    'tgt_pad_idx': train_dataset.fr_vocab['<pad>'],

    'max_sequence_length': MAX_SEQ_LEN,
    'min_token_freq': MIN_TOK_FREQ,
    'seed': 0,
    'log_every': 50,
}

torch.manual_seed(config_mamba['seed'])

config_mamba['train_loader'] = DataLoader(
    train_dataset,
    batch_size=config_mamba['batch_size'],
    shuffle=True,
    collate_fn=lambda batch: generate_batch(batch, config_mamba['src_pad_idx'], config_mamba['tgt_pad_idx'])
)
config_mamba['val_loader'] = DataLoader(
    val_dataset,
    batch_size=config_mamba['batch_size'],
    shuffle=True,
    collate_fn=lambda batch: generate_batch(batch, config_mamba['src_pad_idx'], config_mamba['tgt_pad_idx'])
)

model_mamba = TranslationMamba(
    config_mamba['n_tokens_src'],
    config_mamba['n_tokens_tgt'],
    config_mamba['dim_embedding'],
    config_mamba['n_layers'],
    config_mamba['d_state'],
    config_mamba['d_conv'],
    config_mamba['expand'],
    config_mamba['dropout'],
    config_mamba['src_pad_idx'],
    config_mamba['tgt_pad_idx'],
)

config_mamba['optimizer'] = optim.AdamW(
    model_mamba.parameters(),
    lr=config_mamba['lr'],
    betas=config_mamba['betas'],
    weight_decay=config_mamba['weight_decay'],
)

weight_classes = torch.ones(config_mamba['n_tokens_tgt'], dtype=torch.float)
weight_classes[config_mamba['tgt_vocab']['<unk>']] = 0.1
config_mamba['loss'] = nn.CrossEntropyLoss(
    weight=weight_classes,
    ignore_index=config_mamba['tgt_pad_idx'],
)

print(f"Total params: {sum(p.numel() for p in model_mamba.parameters()):,}")

Total params: 9,201,166


In [38]:
# Entraînement Mamba
!wandb offline

with wandb.init(
        config=config_mamba,
        project='INF8225 - Projet',
        group='Mamba - E1',
        save_code=True,
    ):
    train_model(model_mamba, config_mamba)

wandb: Updated settings file /content/wandb/settings
W&B offline. Running your script from this directory will only write metadata locally. Use `wandb disabled` to completely turn off W&B.


wandb: Loading settings from /content/wandb/settings
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Starting training for 20 epochs, using cuda.

Epoch 1
Train -   loss: 3.11     top-1: 0.44    top-5: 0.62    top-10: 0.69
Eval -    loss: 3.02     top-1: 0.44    top-5: 0.62    top-10: 0.68
I'm only going to show you once.
Je ne suis pas si je vous prie.

Epoch 2
Train -   loss: 2.64     top-1: 0.50    top-5: 0.69    top-10: 0.75
Eval -    loss: 2.52     top-1: 0.51    top-5: 0.70    top-10: 0.75
This tea tastes good.
C'est important.

Epoch 3
Train -   loss: 2.27     top-1: 0.57    top-5: 0.74    top-10: 0.80
Eval -    loss: 2.22     top-1: 0.55    top-5: 0.74    top-10: 0.79
This is worth one million yen.
C'est facile.

Epoch 4
Train -   loss: 1.94     top-1: 0.60    top-5: 0.79    top-10: 0.83
Eval -    loss: 2.00     top-1: 0.59    top-5: 0.78    top-10: 0.82
I know just what to do.
Je sais simplement quoi faire.

Epoch 5
Train -   loss: 1.86     top-1: 0.61    top-5: 0.80    top-10: 0.85
Eval -    loss: 1.83     top-1: 0.62    top-5: 0.80    top-10: 0.85
Have you received the lett

In [39]:
# Sauvegarder le modèle + calculer BLEU
torch.save(model_mamba.state_dict(), 'mamba_main.pt')

bleu_mamba = compute_bleu(model_mamba, val_dataset, config_mamba, n_samples=200)
print(f"BLEU Mamba: {bleu_mamba:.2f}")

BLEU Mamba: 36.19


In [40]:
# Greedy vs Beam search sur Mamba

test_sentences = [
    "I love you.",
    "What time is it?",
    "She is a student.",
    "The weather is nice today.",
    "Can you help me?"
]

results_E2 = {'sentences': [], 'greedy': [], 'beam': [], 'beam_prob': []}

for sent in test_sentences:
    print(f"\nEN: {sent}")
    greedy_pred = greedy_search(
        model_mamba, sent,
        config_mamba['src_vocab'], config_mamba['tgt_vocab'],
        en_tokenizer, device, config_mamba['max_sequence_length'],
    )
    beam_preds = beam_search(
        model_mamba, sent,
        config_mamba['src_vocab'], config_mamba['tgt_vocab'],
        en_tokenizer, device,
        beam_width=10, max_target=100,
        max_sentence_length=config_mamba['max_sequence_length'],
    )
    beam_pred, beam_prob = beam_preds[0]

    print(f'  Greedy: {greedy_pred}')
    print(f'  Beam:   {beam_pred}  ({beam_prob*100:.2f}%)')

    results_E2['sentences'].append(sent)
    results_E2['greedy'].append(greedy_pred)
    results_E2['beam'].append(beam_pred)
    results_E2['beam_prob'].append(beam_prob)

save_results(results_E2, 'E2_decoding')


EN: I love you.
  Greedy: [('<unk> <unk> <unk> <unk>', 0.25726595520973206)]
  Beam:   Je t'adore.  (24.37%)

EN: What time is it?
  Greedy: [('<unk> <unk> <unk> <unk> <unk> <unk> <unk>', 0.2994439899921417)]
  Beam:   Quelle heure est-il là   ?  (28.45%)

EN: She is a student.
  Greedy: [('<unk> <unk> <unk> <unk>', 0.6357073783874512)]
  Beam:   Elle est étudiante.  (60.23%)

EN: The weather is nice today.
  Greedy: [('<unk> <unk> <unk> <unk> <unk>', 0.4290904700756073)]
  Beam:   Il fait beau aujourd'hui.  (40.69%)

EN: Can you help me?
  Greedy: [('<unk> <unk> <unk> <unk> <unk> <unk>', 0.6813371181488037)]
  Beam:   Peux-tu m'aider ?  (64.68%)
Saved pickle: results/E2_decoding.pkl
Saved JSON:   results/E2_decoding.json


In [41]:
# Sauvegarder les résultats E1 (Mamba traduction)
results_E1 = {
    'model_name': 'Mamba',
    'dim_embedding': config_mamba['dim_embedding'],
    'n_layers': config_mamba['n_layers'],
    'd_state': config_mamba['d_state'],
    'bleu': bleu_mamba,
    'total_params': sum(p.numel() for p in model_mamba.parameters()),
}
save_results(results_E1, 'E1_translation')

Saved pickle: results/E1_translation.pkl
Saved JSON:   results/E1_translation.json


## Mamba LM training (synthetic tasks)

In [42]:
class MambaLM(nn.Module):
    """A simple causal language model using Mamba.

    Used for the synthetic tasks (copying, induction).

    Parameters
    ----------
        vocab_size: Size of the token vocabulary.
        d_model: Hidden dimension.
        n_layers: Number of Mamba blocks.
        d_state, d_conv, expand, dropout: Passed to MambaStack.
    """
    def __init__(
            self,
            vocab_size: int,
            d_model: int,
            n_layers: int,
            d_state: int = 16,
            d_conv: int = 4,
            expand: int = 2,
            dropout: float = 0.0,
        ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model).to(device)
        self.mamba = MambaStack(
            d_model=d_model, n_layers=n_layers,
            d_state=d_state, d_conv=d_conv,
            expand=expand, dropout=dropout,
        )
        self.out_head = nn.Linear(d_model, vocab_size).to(device)

    def forward(self, tokens: torch.LongTensor) -> torch.FloatTensor:
        """
        Args
        ----
            tokens: Input tokens.
                Shape of [batch_size, seq_len].

        Output
        ------
            logits: Shape of [batch_size, seq_len, vocab_size].
        """
        x = self.embedding(tokens)
        x = self.mamba(x)
        return self.out_head(x)

In [43]:
class SelectiveCopyingDataset(Dataset):
    """Selective Copying task from Mamba paper, Table 1.

    Builds sequences of the form
        [noise, ..., t_1, noise, t_2, ..., MARKER, t_1, t_2, ...]
    where n_to_copy real tokens are hidden at random positions in a
    noisy body and must be reproduced in order after a marker token.

    A non-selective SSM (S4) cannot solve this task because the spacing
    between real tokens is random and content-dependent filtering is
    needed.

    Parameters
    ----------
        num_samples: Dataset size.
        seq_len: Length of the noisy body.
        n_to_copy: Number of real tokens to copy.
        vocab_size: Token 0 = noise, last token = marker.
        seed: RNG seed for reproducibility.
    """
    NOISE_TOKEN = 0

    def __init__(
            self,
            num_samples: int = 5000,
            seq_len: int = 256,
            n_to_copy: int = 16,
            vocab_size: int = 16,
            seed: int = 42,
        ):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.n_to_copy = n_to_copy
        self.vocab_size = vocab_size
        self.MARKER = vocab_size - 1
        self.seed = seed

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        g = torch.Generator().manual_seed(self.seed + idx)

        # Real tokens to copy (range [1, vocab_size-2], no noise or marker)
        tokens_to_copy = torch.randint(1, self.vocab_size - 1, (self.n_to_copy,), generator=g)

        # Distinct positions in the body, sorted
        positions = torch.randperm(self.seq_len, generator=g)[:self.n_to_copy]
        positions = positions.sort().values

        body = torch.full((self.seq_len,), self.NOISE_TOKEN, dtype=torch.long)
        body[positions] = tokens_to_copy

        full = torch.cat([
            body,
            torch.tensor([self.MARKER], dtype=torch.long),
            tokens_to_copy,
        ])

        input_ids = full[:-1]
        target_ids = full[1:]

        # Loss mask: only the last n_to_copy positions (the answer)
        mask = torch.zeros_like(target_ids, dtype=torch.bool)
        mask[-self.n_to_copy:] = True

        return input_ids, target_ids, mask

In [44]:
class InductionHeadsDataset(Dataset):
    """Induction Heads task, Figure 2 (right) of Mamba paper.

    Builds sequences of the form
        [rand, ..., MARKER, x, ..., rand, MARKER]
    At the final marker the model must predict x, i.e. the token that
    immediately followed the first marker. Tests associative recall and
    length extrapolation (train at short len, test at much longer).

    Parameters
    ----------
        num_samples: Dataset size.
        seq_len: Total sequence length.
        vocab_size: Last token is the marker.
        seed: RNG seed.
    """
    def __init__(
            self,
            num_samples: int = 5000,
            seq_len: int = 256,
            vocab_size: int = 16,
            seed: int = 42,
        ):
        assert seq_len >= 5, 'seq_len must be at least 5'
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.MARKER = vocab_size - 1
        self.seed = seed

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        g = torch.Generator().manual_seed(self.seed + idx)

        # full has length seq_len+1, input = full[:-1] has length seq_len
        body_len = self.seq_len - 1
        body = torch.randint(1, self.vocab_size - 1, (body_len,), generator=g)

        # Place [MARKER, x] at a random position (not near the end)
        first_pos = torch.randint(0, body_len - 2, (1,), generator=g).item()
        x = torch.randint(1, self.vocab_size - 1, (1,), generator=g).item()
        body[first_pos] = self.MARKER
        body[first_pos + 1] = x

        full = torch.cat([body, torch.tensor([self.MARKER, x], dtype=torch.long)])
        input_ids = full[:-1]
        target_ids = full[1:]

        # Loss mask: only the last position
        mask = torch.zeros_like(target_ids, dtype=torch.bool)
        mask[-1] = True

        return input_ids, target_ids, mask



In [45]:
def train_synthetic(
        model: nn.Module,
        train_dataset: Dataset,
        val_dataset: Dataset,
        num_steps: int = 2000,
        batch_size: int = 32,
        lr: float = 1e-3,
        log_every: int = 100,
        eval_every: int = 200,
    ) -> dict:
    """Train a MambaLM on a synthetic task.

    Loss is computed only on the positions marked True in the mask.
    Returns a dict with step / train_loss / val_acc history.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    history = {'step': [], 'train_loss': [], 'val_acc': []}
    model.train()
    step = 0
    running_loss = []
    train_iter = iter(train_loader)

    print(f'Training {type(model).__name__} for {num_steps} steps...')
    while step < num_steps:
        try:
            input_ids, target_ids, mask = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            input_ids, target_ids, mask = next(train_iter)

        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        mask = mask.to(device)

        logits = model(input_ids)
        V = logits.size(-1)

        # Cross-entropy on masked positions only
        logits_flat = logits.reshape(-1, V)
        target_flat = target_ids.reshape(-1)
        mask_flat = mask.reshape(-1)
        loss = F.cross_entropy(logits_flat[mask_flat], target_flat[mask_flat])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss.append(loss.item())
        step += 1

        if step % log_every == 0:
            avg_loss = float(np.mean(running_loss[-log_every:]))
            print(f'  step {step:5d} | loss={avg_loss:.4f}')

        if step % eval_every == 0:
            acc = evaluate_synthetic(model, val_loader)
            history['step'].append(step)
            history['train_loss'].append(float(np.mean(running_loss[-eval_every:])))
            history['val_acc'].append(acc)
            print(f'           | val_acc={acc:.4f}')
            model.train()

    return history

In [46]:

@torch.no_grad()
def evaluate_synthetic(model: nn.Module, loader: DataLoader) -> float:
    """Accuracy over masked positions only."""
    model.eval()
    correct, total = 0, 0
    for input_ids, target_ids, mask in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        mask = mask.to(device)

        logits = model(input_ids)
        preds = logits.argmax(dim=-1)

        correct += ((preds == target_ids) & mask).sum().item()
        total += mask.sum().item()
    return correct / max(total, 1)


In [47]:
# Selective Copying
VOCAB_SYN = 16
SEQ_SYN = 128
N_COPY = 8

train_ds_sc = SelectiveCopyingDataset(
    num_samples=3000, seq_len=SEQ_SYN, n_to_copy=N_COPY, vocab_size=VOCAB_SYN,
)
val_ds_sc = SelectiveCopyingDataset(
    num_samples=500, seq_len=SEQ_SYN, n_to_copy=N_COPY, vocab_size=VOCAB_SYN, seed=99,
)

torch.manual_seed(0)
model_E3 = MambaLM(vocab_size=VOCAB_SYN, d_model=64, n_layers=2).to(device)
hist_E3 = train_synthetic(
    model_E3, train_ds_sc, val_ds_sc,
    num_steps=1500, batch_size=32, lr=1e-3,
    log_every=300, eval_every=500,
)

Training MambaLM for 1500 steps...
  step   300 | loss=2.6653
           | val_acc=0.1323
  step   600 | loss=2.5910
  step   900 | loss=1.5601
           | val_acc=0.8217
  step  1200 | loss=0.4829
  step  1500 | loss=0.2149
           | val_acc=0.9670


In [48]:
# Sauvegarder les résultats E3
results_E3 = {
    'task': 'selective_copying',
    'seq_len': SEQ_SYN,
    'n_to_copy': N_COPY,
    'final_val_acc': hist_E3['val_acc'][-1] if hist_E3['val_acc'] else 0.0,
    'history': hist_E3,
}
save_results(results_E3, 'E3_selective_copying')
print(f"E3 final accuracy: {results_E3['final_val_acc']:.4f}")

Saved pickle: results/E3_selective_copying.pkl
Saved JSON:   results/E3_selective_copying.json
E3 final accuracy: 0.9670


In [49]:
@torch.no_grad()
def evaluate_length_extrapolation(
        model: nn.Module,
        lengths: list,
        vocab_size: int = 16,
        n_samples_per_length: int = 200,
    ) -> dict:
    """Evaluate induction-heads accuracy at multiple sequence lengths.

    Returns a dict {seq_len: accuracy}.
    """
    results = {}
    for L in lengths:
        dataset = InductionHeadsDataset(
            num_samples=n_samples_per_length,
            seq_len=L,
            vocab_size=vocab_size,
            seed=9999,
        )
        loader = DataLoader(dataset, batch_size=8)
        acc = evaluate_synthetic(model, loader)
        results[L] = acc
        print(f'  seq_len={L:>7d}  acc={acc:.4f}')
    return results

In [50]:
# Induction Heads avec extrapolation en longueur
train_ds_ih = InductionHeadsDataset(num_samples=3000, seq_len=128, vocab_size=VOCAB_SYN)
val_ds_ih = InductionHeadsDataset(num_samples=500, seq_len=128, vocab_size=VOCAB_SYN, seed=99)

torch.manual_seed(0)
model_E4 = MambaLM(vocab_size=VOCAB_SYN, d_model=64, n_layers=2).to(device)
hist_E4 = train_synthetic(
    model_E4, train_ds_ih, val_ds_ih,
    num_steps=2000, batch_size=32, lr=1e-3,
    log_every=400, eval_every=500,
)

# Length extrapolation test
test_lengths = [64, 128, 256, 512, 1024, 2048]
extrap_E4 = evaluate_length_extrapolation(
    model_E4, test_lengths, vocab_size=VOCAB_SYN, n_samples_per_length=100,
)

Training MambaLM for 2000 steps...
  step   400 | loss=2.6360
           | val_acc=0.1260
  step   800 | loss=2.4886
           | val_acc=0.5300
  step  1200 | loss=1.2897
           | val_acc=1.0000
  step  1600 | loss=0.0052
  step  2000 | loss=0.0018
           | val_acc=1.0000
  seq_len=     64  acc=1.0000
  seq_len=    128  acc=1.0000
  seq_len=    256  acc=1.0000
  seq_len=    512  acc=1.0000
  seq_len=   1024  acc=0.9500
  seq_len=   2048  acc=0.8600


In [51]:
# Sauvegarder les résultats E4
results_E4 = {
    'task': 'induction_heads',
    'train_len': 128,
    'final_val_acc': hist_E4['val_acc'][-1] if hist_E4['val_acc'] else 0.0,
    'history': hist_E4,
    'extrapolation': extrap_E4,
}
save_results(results_E4, 'E4_induction_heads')
print(f"E4 training accuracy: {results_E4['final_val_acc']:.4f}")
print(f"Length extrapolation: {extrap_E4}")

Saved pickle: results/E4_induction_heads.pkl
Saved JSON:   results/E4_induction_heads.json
E4 training accuracy: 1.0000
Length extrapolation: {64: 1.0, 128: 1.0, 256: 1.0, 512: 1.0, 1024: 0.95, 2048: 0.86}


## Variantes de selectivite et init of A

In [52]:
class SelectiveSSMAblatable(nn.Module):
    """Selective SSM with flags to turn off each selective parameter.

    When a parameter is NOT selective, it becomes a learned constant
    (shared across the sequence), recovering an LTI (S4-like) behavior.
    Used to reproduce Table 7 of the Mamba paper.

    Parameters
    ----------
        d_inner, d_state: Same as SelectiveSSM.
        selective_delta: If False, Delta is a fixed learned parameter.
        selective_B: If False, B is a fixed learned parameter.
        selective_C: If False, C is a fixed learned parameter.
        A_init: One of 's4d_real', 's4d_complex', 'random'.
        dt_min, dt_max: Same as SelectiveSSM.
    """
    def __init__(
            self,
            d_inner: int,
            d_state: int = 16,
            selective_delta: bool = True,
            selective_B: bool = True,
            selective_C: bool = True,
            A_init: str = 's4d_real',
            dt_min: float = 0.001,
            dt_max: float = 0.1,
        ):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        self.selective_delta = selective_delta
        self.selective_B = selective_B
        self.selective_C = selective_C

        # Input projection - only for the selective parts
        proj_dims = []
        if selective_delta:
            self.dt_rank = max(1, d_inner // 32)
            proj_dims.append(self.dt_rank)
        if selective_B:
            proj_dims.append(d_state)
        if selective_C:
            proj_dims.append(d_state)
        self.proj_dims = proj_dims

        if len(proj_dims) > 0:
            self.x_proj = nn.Linear(d_inner, sum(proj_dims), bias=False).to(device)
        else:
            self.x_proj = None

        # Delta handling
        if selective_delta:
            self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True).to(device)
            dt_init_std = self.dt_rank ** -0.5
            nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
            dt = torch.exp(
                torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min))
                + math.log(dt_min)
            )
            inv_dt = dt + torch.log(-torch.expm1(-dt))
            with torch.no_grad():
                self.dt_proj.bias.copy_(inv_dt)
        else:
            # Fixed Delta per channel (pre-softplus value)
            init = torch.log(torch.exp(torch.ones(d_inner) * 0.01) - 1)
            self.delta_param = nn.Parameter(init.to(device))

        # B handling
        if not selective_B:
            self.B_param = nn.Parameter(torch.randn(d_state).to(device) * 0.1)

        # C handling
        if not selective_C:
            self.C_param = nn.Parameter(torch.randn(d_state).to(device) * 0.1)

        # A initialization (Table 8)
        if A_init == 's4d_real':
            A = repeat(
                torch.arange(1, d_state + 1, dtype=torch.float),
                'n -> d n', d=d_inner,
            )
        elif A_init == 's4d_complex':
            # Real approximation of S4D-Lin: A_n = -0.5 + n*i, we use |A_n|
            n_range = torch.arange(1, d_state + 1, dtype=torch.float)
            magnitudes = torch.sqrt(0.25 + n_range ** 2)
            A = repeat(magnitudes, 'n -> d n', d=d_inner)
        elif A_init == 'random':
            A = torch.exp(torch.randn(d_inner, d_state))
        else:
            raise ValueError(f'Unknown A_init: {A_init}')

        self.A_log = nn.Parameter(torch.log(A).to(device))
        self.D = nn.Parameter(torch.ones(d_inner).to(device))

    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        """Same interface as SelectiveSSM.

        Args
        ----
            x: Shape of [batch_size, seq_len, d_inner].

        Output
        ------
            y: Shape of [batch_size, seq_len, d_inner].
        """
        batch, sl, _ = x.shape
        A = -torch.exp(self.A_log)

        # Retrieve the selective parts from x
        if self.x_proj is not None:
            x_dbl = self.x_proj(x)
            parts = torch.split(x_dbl, self.proj_dims, dim=-1)
        else:
            parts = []

        # Walk through parts in the order they were concatenated
        ptr = 0
        if self.selective_delta:
            delta_raw = parts[ptr]; ptr += 1
            delta = F.softplus(self.dt_proj(delta_raw))
        else:
            delta = F.softplus(self.delta_param)
            delta = delta.view(1, 1, -1).expand(batch, sl, -1)

        if self.selective_B:
            B = parts[ptr]; ptr += 1
        else:
            B = self.B_param.view(1, 1, -1).expand(batch, sl, -1)

        if self.selective_C:
            C = parts[ptr]; ptr += 1
        else:
            C = self.C_param.view(1, 1, -1).expand(batch, sl, -1)

        # Discretize and scan (same as SelectiveSSM)
        delta_A = torch.exp(torch.einsum('bld,dn->bldn', delta, A))
        delta_B_x = torch.einsum('bld,bln,bld->bldn', delta, B, x)

        h = torch.zeros(batch, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(sl):
            h = delta_A[:, t] * h + delta_B_x[:, t]
            y_t = torch.einsum('bdn,bn->bd', h, C[:, t])
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        y = y + x * self.D
        return y

In [53]:
class MambaBlockAblatable(nn.Module):
    """MambaBlock with pluggable SelectiveSSMAblatable for ablation runs.

    Parameters
    ----------
        d_model, d_state, d_conv, expand: Same as MambaBlock.
        selective_delta, selective_B, selective_C: Passed to the SSM.
        A_init: Passed to the SSM.
    """
    def __init__(
            self,
            d_model: int,
            d_state: int = 16,
            d_conv: int = 4,
            expand: int = 2,
            selective_delta: bool = True,
            selective_B: bool = True,
            selective_C: bool = True,
            A_init: str = 's4d_real',
        ):
        super().__init__()

        self.d_inner = expand * d_model
        self.d_conv = d_conv

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False).to(device)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, groups=self.d_inner,
            padding=d_conv - 1, bias=True,
        ).to(device)
        self.ssm = SelectiveSSMAblatable(
            d_inner=self.d_inner, d_state=d_state,
            selective_delta=selective_delta,
            selective_B=selective_B,
            selective_C=selective_C,
            A_init=A_init,
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False).to(device)

    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        sl = x.size(1)
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        x_conv = rearrange(x_ssm, 'b l d -> b d l')
        x_conv = self.conv1d(x_conv)[:, :, :sl]
        x_conv = rearrange(x_conv, 'b d l -> b l d')
        x_conv = F.silu(x_conv)

        y = self.ssm(x_conv)
        y = y * F.silu(z)
        return self.out_proj(y)

In [54]:
# Configs for Table 7 of the Mamba paper (name, sel_delta, sel_B, sel_C)
ABLATION_CONFIGS_TABLE_7 = [
    ('All non-selective (S4-like)', False, False, False),
    ('Selective B only',            False, True,  False),
    ('Selective C only',            False, False, True ),
    ('Selective Delta only',        True,  False, False),
    ('Full Mamba (Delta, B, C)',    True,  True,  True ),
]

# A initialization variants (Table 8)
ABLATION_CONFIGS_TABLE_8 = ['s4d_real', 's4d_complex', 'random']


In [55]:
# Ablation Δ, B, C (Table 7)

class MambaLMAblatable(nn.Module):
    """MambaLM utilisant MambaBlockAblatable pour les runs d'ablation."""
    def __init__(
            self, vocab_size, d_model, n_layers,
            selective_delta=True, selective_B=True, selective_C=True,
            A_init='s4d_real', d_state=16, d_conv=4, expand=2, dropout=0.0,
        ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model).to(device)
        self.layers = nn.ModuleList([
            MambaBlockAblatable(
                d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand,
                selective_delta=selective_delta,
                selective_B=selective_B,
                selective_C=selective_C,
                A_init=A_init,
            )
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.out_head = nn.Linear(d_model, vocab_size).to(device)

    def forward(self, tokens):
        x = self.embedding(tokens)
        for block, norm in zip(self.layers, self.norms):
            x = x + self.dropout(block(norm(x)))
        x = self.final_norm(x)
        return self.out_head(x)

In [56]:
#  Ablation sélective
results_E5 = {'configs': []}
for name, sd, sB, sC in ABLATION_CONFIGS_TABLE_7:
    print(f"\n--- {name} ---")
    torch.manual_seed(0)
    model_abl = MambaLMAblatable(
        vocab_size=VOCAB_SYN, d_model=64, n_layers=2,
        selective_delta=sd, selective_B=sB, selective_C=sC,
    ).to(device)
    hist = train_synthetic(
        model_abl, train_ds_sc, val_ds_sc,
        num_steps=1500, batch_size=32, lr=1e-3,
        log_every=300, eval_every=500,
    )
    results_E5['configs'].append({
        'name': name,
        'final_val_acc': hist['val_acc'][-1] if hist['val_acc'] else 0.0,
        'history': hist,
    })


--- All non-selective (S4-like) ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6633
           | val_acc=0.1258
  step   600 | loss=2.6073
  step   900 | loss=2.5578
           | val_acc=0.1750
  step  1200 | loss=2.4862
  step  1500 | loss=2.3874
           | val_acc=0.2535

--- Selective B only ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6658
           | val_acc=0.1237
  step   600 | loss=2.6160
  step   900 | loss=2.5672
           | val_acc=0.1820
  step  1200 | loss=2.4443
  step  1500 | loss=1.7351
           | val_acc=0.4995

--- Selective C only ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6658
           | val_acc=0.1225
  step   600 | loss=2.6108
  step   900 | loss=2.5603
           | val_acc=0.1720
  step  1200 | loss=2.4921
  step  1500 | loss=2.3934
           | val_acc=0.2485

--- Selective Delta only ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6670
           | val_acc=

In [57]:
# Sauvegarder les résultats E5
save_results(results_E5, 'E5_selectivity_ablation')
for cfg in results_E5['configs']:
    print(f"  {cfg['name']:32s} val_acc={cfg['final_val_acc']:.4f}")

Saved pickle: results/E5_selectivity_ablation.pkl
Saved JSON:   results/E5_selectivity_ablation.json
  All non-selective (S4-like)      val_acc=0.2535
  Selective B only                 val_acc=0.4995
  Selective C only                 val_acc=0.2485
  Selective Delta only             val_acc=0.3020
  Full Mamba (Delta, B, C)         val_acc=0.9670


In [58]:
# Balayage d_state
results_E6 = {'d_state_values': [], 'val_acc': [], 'n_params': []}
for N in [4, 8, 16, 32]:
    print(f"\n--- d_state = {N} ---")
    torch.manual_seed(0)
    model_N = MambaLM(vocab_size=VOCAB_SYN, d_model=64, n_layers=2, d_state=N).to(device)
    hist = train_synthetic(model_N, train_ds_sc, val_ds_sc, num_steps=1500, batch_size=32, lr=1e-3, log_every=300, eval_every=500)
    results_E6['d_state_values'].append(N)
    results_E6['val_acc'].append(hist['val_acc'][-1] if hist['val_acc'] else 0.0)
    results_E6['n_params'].append(sum(p.numel() for p in model_N.parameters()))


--- d_state = 4 ---
Training MambaLM for 1500 steps...
  step   300 | loss=2.6673
           | val_acc=0.1200
  step   600 | loss=2.6099
  step   900 | loss=1.8786
           | val_acc=0.6905
  step  1200 | loss=0.7745
  step  1500 | loss=0.2739
           | val_acc=0.9503

--- d_state = 8 ---
Training MambaLM for 1500 steps...
  step   300 | loss=2.6625
           | val_acc=0.1123
  step   600 | loss=2.6093
  step   900 | loss=1.9151
           | val_acc=0.6757
  step  1200 | loss=0.8107
  step  1500 | loss=0.3672
           | val_acc=0.8670

--- d_state = 16 ---
Training MambaLM for 1500 steps...
  step   300 | loss=2.6653
           | val_acc=0.1323
  step   600 | loss=2.5910
  step   900 | loss=1.5601
           | val_acc=0.8217
  step  1200 | loss=0.4829
  step  1500 | loss=0.2149
           | val_acc=0.9670

--- d_state = 32 ---
Training MambaLM for 1500 steps...
  step   300 | loss=2.6658
           | val_acc=0.1353
  step   600 | loss=2.6030
  step   900 | loss=2.3973
        

In [59]:
# Sauvegarder les résultats E6
save_results(results_E6, 'E6_state_size')
for N, acc, params in zip(results_E6['d_state_values'], results_E6['val_acc'], results_E6['n_params']):
    print(f"  d_state={N:3d}  val_acc={acc:.4f}  params={params:,}")

Saved pickle: results/E6_state_size.pkl
Saved JSON:   results/E6_state_size.json
  d_state=  4  val_acc=0.9503  params=58,320
  d_state=  8  val_acc=0.8670  params=61,392
  d_state= 16  val_acc=0.9670  params=67,536
  d_state= 32  val_acc=0.7835  params=79,824


In [60]:
# Ablation initialisation de A
results_E7 = {'inits': []}
for A_init in ['s4d_real', 's4d_complex', 'random']:
    print(f"\n--- A_init = {A_init} ---")
    torch.manual_seed(0)
    model_A = MambaLMAblatable(
        vocab_size=VOCAB_SYN, d_model=64, n_layers=2, A_init=A_init,
    ).to(device)
    hist = train_synthetic(model_A, train_ds_sc, val_ds_sc, num_steps=1500, batch_size=32, lr=1e-3, log_every=300, eval_every=500)
    results_E7['inits'].append({
        'A_init': A_init,
        'final_val_acc': hist['val_acc'][-1] if hist['val_acc'] else 0.0,
    })


--- A_init = s4d_real ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6653
           | val_acc=0.1323
  step   600 | loss=2.5910
  step   900 | loss=1.5601
           | val_acc=0.8217
  step  1200 | loss=0.4829
  step  1500 | loss=0.2149
           | val_acc=0.9670

--- A_init = s4d_complex ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6653
           | val_acc=0.1330
  step   600 | loss=2.5918
  step   900 | loss=1.5985
           | val_acc=0.8303
  step  1200 | loss=0.5028
  step  1500 | loss=0.2179
           | val_acc=0.9647

--- A_init = random ---
Training MambaLMAblatable for 1500 steps...
  step   300 | loss=2.6651
           | val_acc=0.1380
  step   600 | loss=2.5569
  step   900 | loss=1.3494
           | val_acc=0.8750
  step  1200 | loss=0.3846
  step  1500 | loss=0.1531
           | val_acc=0.9780


In [61]:
# Sauvegarder les résultats E7
save_results(results_E7, 'E7_A_init')
for init in results_E7['inits']:
    print(f"  A_init={init['A_init']:15s}  val_acc={init['final_val_acc']:.4f}")

Saved pickle: results/E7_A_init.pkl
Saved JSON:   results/E7_A_init.json
  A_init=s4d_real         val_acc=0.9670
  A_init=s4d_complex      val_acc=0.9647
  A_init=random           val_acc=0.9780


## **BenchMarking**

In [62]:
class MambaLMForBenchmark(nn.Module):
    """Small Mamba LM used only for benchmarking."""
    def __init__(self, vocab_size=1000, d_model=128, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model).to(device)
        self.body  = MambaStack(d_model=d_model, n_layers=n_layers, dropout=0.0)
        self.head  = nn.Linear(d_model, vocab_size).to(device)

    def forward(self, x):
        return self.head(self.body(self.embed(x)))

In [63]:
class TinyTransformerLM(nn.Module):
    """Decoder-only Transformer for benchmarking, using nn.TransformerEncoderLayer
    with a causal mask.
    """
    def __init__(self, vocab_size=1000, d_model=128, n_layers=4, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model).to(device)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=4 * d_model,
            dropout=0.0, batch_first=True, activation='gelu',
        )
        self.body = nn.TransformerEncoder(layer, num_layers=n_layers).to(device)
        self.head = nn.Linear(d_model, vocab_size).to(device)

    def forward(self, x):
        sl = x.size(1)
        mask = torch.triu(
            torch.ones(sl, sl, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        h = self.body(self.embed(x), mask=mask, is_causal=True)
        return self.head(h)

In [64]:
class TinyGRULM(nn.Module):
    """Simple GRU LM for benchmarking, using the built-in nn.GRU (we are
    benchmarking asymptotic complexity, not custom implementation).
    """
    def __init__(self, vocab_size=1000, d_model=128, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model).to(device)
        self.gru = nn.GRU(
            input_size=d_model, hidden_size=d_model,
            num_layers=n_layers, batch_first=True, dropout=0.0,
        ).to(device)
        self.head = nn.Linear(d_model, vocab_size).to(device)

    def forward(self, x):
        h, _ = self.gru(self.embed(x))
        return self.head(h)

In [65]:
def benchmark_model(
        make_model,
        seq_lengths: list,
        batch_size: int = 4,
        vocab_size: int = 1000,
        n_warmup: int = 2,
        n_trials: int = 3,
        include_backward: bool = True,
    ) -> dict:
    """Benchmark a model across multiple sequence lengths.

    Args
    ----
        make_model: zero-arg callable that instantiates a fresh model.
        seq_lengths: list of seq_len values to sweep.
        batch_size, n_warmup, n_trials: benchmarking knobs.
        include_backward: if True, also measure backward pass time.

    Output
    ------
        results dict with keys: 'seq_len', 'fwd_ms', 'bwd_ms', 'peak_mb', 'oom'.
    """
    results = {
        'seq_len': [], 'fwd_ms': [], 'bwd_ms': [],
        'peak_mb': [], 'oom': [],
    }

    for L in seq_lengths:
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        try:
            model = make_model().to(device)
            model.train()

            # Warmup
            for _ in range(n_warmup):
                x = torch.randint(0, vocab_size, (batch_size, L), device=device)
                out = model(x)
                if include_backward:
                    out.sum().backward()
                    model.zero_grad()
                del out, x
            if device.type == 'cuda':
                torch.cuda.synchronize()

            # Forward only
            fwd_times = []
            for _ in range(n_trials):
                x = torch.randint(0, vocab_size, (batch_size, L), device=device)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
                with torch.no_grad():
                    out = model(x)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                fwd_times.append(time.time() - t0)
                del out, x

            # Forward + backward
            bwd_times = []
            if include_backward:
                if device.type == 'cuda':
                    torch.cuda.reset_peak_memory_stats()
                for _ in range(n_trials):
                    x = torch.randint(0, vocab_size, (batch_size, L), device=device)
                    if device.type == 'cuda':
                        torch.cuda.synchronize()
                    t0 = time.time()
                    out = model(x)
                    out.sum().backward()
                    if device.type == 'cuda':
                        torch.cuda.synchronize()
                    bwd_times.append(time.time() - t0)
                    model.zero_grad()
                    del out, x

            peak_mb = (
                torch.cuda.max_memory_allocated() / 1e6
                if device.type == 'cuda' else 0.0
            )

            results['seq_len'].append(L)
            results['fwd_ms'].append(np.mean(fwd_times) * 1000)
            results['bwd_ms'].append(
                np.mean(bwd_times) * 1000 if include_backward else 0.0
            )
            results['peak_mb'].append(peak_mb)
            results['oom'].append(False)

            print(
                f'  L={L:>5d} | '
                f'fwd={np.mean(fwd_times)*1000:7.2f}ms | '
                f'bwd={np.mean(bwd_times)*1000:7.2f}ms | '
                f'peak={peak_mb:7.1f}MB'
            )

            del model
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

        except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
            if 'out of memory' in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
                print(f'  L={L:>5d} | OUT OF MEMORY')
                results['seq_len'].append(L)
                results['fwd_ms'].append(float('nan'))
                results['bwd_ms'].append(float('nan'))
                results['peak_mb'].append(float('nan'))
                results['oom'].append(True)
                gc.collect()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
            else:
                raise

    return results

In [66]:
def run_full_benchmark(
        seq_lengths: list = None,
        d_model: int = 128,
        n_layers: int = 4,
        batch_size: int = 4,
    ) -> dict:
    """Run the full benchmark: Mamba vs Transformer vs GRU.

    Returns a dict {'mamba': ..., 'transformer': ..., 'gru': ...}.
    """
    if seq_lengths is None:
        seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192]

    print(f'Efficiency benchmark (d_model={d_model}, n_layers={n_layers}, batch_size={batch_size}, device={device})')

    def count_params(model):
        return sum(p.numel() for p in model.parameters())

    # Print param counts for fairness check
    m = MambaLMForBenchmark(d_model=d_model, n_layers=n_layers)
    print(f'  Mamba:       {count_params(m):>10,d} params')
    del m
    t = TinyTransformerLM(d_model=d_model, n_layers=n_layers)
    print(f'  Transformer: {count_params(t):>10,d} params')
    del t
    g = TinyGRULM(d_model=d_model, n_layers=n_layers)
    print(f'  GRU:         {count_params(g):>10,d} params')
    del g
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    results = {}

    print('\nBenchmarking Mamba...')
    results['mamba'] = benchmark_model(
        lambda: MambaLMForBenchmark(d_model=d_model, n_layers=n_layers),
        seq_lengths, batch_size=batch_size,
    )

    print('\nBenchmarking Transformer...')
    results['transformer'] = benchmark_model(
        lambda: TinyTransformerLM(d_model=d_model, n_layers=n_layers),
        seq_lengths, batch_size=batch_size,
    )

    print('\nBenchmarking GRU...')
    results['gru'] = benchmark_model(
        lambda: TinyGRULM(d_model=d_model, n_layers=n_layers),
        seq_lengths, batch_size=batch_size,
    )

    return results

In [67]:
def pretty_print_results(results: dict):
    """Print a side-by-side table comparing the three models."""
    seq_lens = results['mamba']['seq_len']
    print(f'\n{"Seq Len":>8} | {"Mamba fwd":>11} | {"Trans fwd":>11} | {"GRU fwd":>11} | {"Mamba mem":>11} | {"Trans mem":>11}')
    print('-' * 80)
    for i, L in enumerate(seq_lens):
        def fmt(v, unit='ms'):
            if isinstance(v, float) and np.isnan(v):
                return 'OOM'.rjust(11)
            if v is None:
                return '-'.rjust(11)
            return f'{v:>8.1f} {unit}'
        print(
            f'{L:>8d} | '
            f'{fmt(results["mamba"]["fwd_ms"][i]):>11} | '
            f'{fmt(results["transformer"]["fwd_ms"][i]):>11} | '
            f'{fmt(results["gru"]["fwd_ms"][i]):>11} | '
            f'{fmt(results["mamba"]["peak_mb"][i], "MB"):>11} | '
            f'{fmt(results["transformer"]["peak_mb"][i], "MB"):>11}'
        )

In [68]:
# Lancer le benchmark complet
results_E11 = run_full_benchmark(
    seq_lengths=[128, 256, 512, 1024, 2048, 4096, 8192],
    d_model=128, n_layers=4, batch_size=4,
)
pretty_print_results(results_E11)

Efficiency benchmark (d_model=128, n_layers=4, batch_size=4, device=cuda)
  Mamba:          723,560 params
  Transformer:  1,050,088 params
  GRU:            653,288 params

Benchmarking Mamba...
  L=  128 | fwd=  61.49ms | bwd= 268.72ms | peak=  293.5MB
  L=  256 | fwd= 121.04ms | bwd= 530.62ms | peak=  416.6MB
  L=  512 | fwd= 235.44ms | bwd=1064.52ms | peak=  659.9MB
  L= 1024 | fwd= 452.26ms | bwd=2105.71ms | peak= 1144.4MB
  L= 2048 | fwd= 902.60ms | bwd=6561.25ms | peak= 2112.1MB
  L= 4096 | fwd=1796.43ms | bwd=22776.07ms | peak= 4053.8MB
  L= 8192 | fwd=3586.36ms | bwd=84368.63ms | peak= 7927.6MB

Benchmarking Transformer...
  L=  128 | fwd=   2.74ms | bwd=   9.19ms | peak=  195.6MB
  L=  256 | fwd=   2.73ms | bwd=   9.68ms | peak=  218.6MB
  L=  512 | fwd=   2.78ms | bwd=   9.74ms | peak=  265.0MB
  L= 1024 | fwd=   2.91ms | bwd=   9.76ms | peak=  355.7MB
  L= 2048 | fwd=   5.15ms | bwd=  16.26ms | peak=  533.1MB
  L= 4096 | fwd=  12.50ms | bwd=  38.68ms | peak=  893.4MB
  L= 8

In [69]:
# Sauvegarder les résultats E11
save_results(results_E11, 'E11_efficiency_benchmark')

Saved pickle: results/E11_efficiency_benchmark.pkl
Saved JSON:   results/E11_efficiency_benchmark.json


## **Sweeps hyperparamètres Mamba (E8, E9, E10)**

Balayages de 3 hyperparamètres Mamba sur la tâche de traduction : `n_layers` (E8), `expand` (E9), `dropout` (E10). On utilise la config FAST (dim=128, 10 epochs)

In [70]:
# Fonction générique de sweep d'hyperparamètre sur la traduction
import copy
import time

def run_translation_sweep(
        param_name: str,
        values: list,
        base_config: dict,
        train_dataset,
        val_dataset,
        train_model_fn,
        compute_bleu_fn=None,
        n_bleu_samples: int = 200,
    ) -> dict:
    """Sweep d'un hyperparamètre Mamba sur la tâche de traduction."""
    print(f'Translation sweep: {param_name} in {values}')

    results = {
        'param_name': param_name, 'values': [],
        'val_loss': [], 'top1': [], 'bleu': [], 'training_time_s': [],
    }

    for val in values:
        print(f'\n--- {param_name} = {val} ---')

        model_kwargs = {
            'n_tokens_src': len(train_dataset.en_vocab),
            'n_tokens_tgt': len(train_dataset.fr_vocab),
            'dim_embedding': base_config['dim_embedding'],
            'n_layers': base_config['n_layers'],
            'd_state': base_config['d_state'],
            'd_conv': base_config['d_conv'],
            'expand': base_config['expand'],
            'dropout': base_config['dropout'],
            'src_pad_idx': train_dataset.en_vocab['<pad>'],
            'tgt_pad_idx': train_dataset.fr_vocab['<pad>'],
        }
        model_kwargs[param_name] = val

        torch.manual_seed(0)
        model = TranslationMamba(**model_kwargs).to(device)

        config_run = copy.deepcopy(base_config)
        config_run[param_name] = val
        config_run['optimizer'] = torch.optim.AdamW(
            model.parameters(),
            lr=config_run['lr'], betas=config_run['betas'],
            weight_decay=config_run['weight_decay'],
        )

        t0 = time.time()
        train_model_fn(model, config_run)
        elapsed = time.time() - t0

        bleu_score = float('nan')
        if compute_bleu_fn is not None:
            print('  Computing BLEU...')
            bleu_score = compute_bleu_fn(
                model, val_dataset, config_run, n_samples=n_bleu_samples,
            )
            print(f'  -> BLEU = {bleu_score:.2f}')

        results['values'].append(val)
        results['val_loss'].append(float('nan'))
        results['top1'].append(float('nan'))
        results['bleu'].append(bleu_score)
        results['training_time_s'].append(elapsed)

    return results

In [71]:
# Config FAST : modèle plus petit (we are evaluating as of now on 10 epochs)

config_fast = dict(config_mamba)
config_fast.update(MAMBA_TRANSLATION_FAST)
config_fast['train_loader'] = config_mamba['train_loader']
config_fast['val_loader']   = config_mamba['val_loader']
config_fast['loss']         = config_mamba['loss']

print("Config FAST prête :")
print(f"  dim_embedding = {config_fast['dim_embedding']}")
print(f"  n_layers      = {config_fast['n_layers']}")
print(f"  epochs        = {config_fast['epochs']}")
print(f"  batch_size    = {config_fast['batch_size']}")

Config FAST prête :
  dim_embedding = 128
  n_layers      = 3
  epochs        = 10
  batch_size    = 256


In [72]:
# E8 : Sweep du nombre de couches Mamba
# Using valeurs : [2, 4, 6]

results_E8 = run_translation_sweep(
    'n_layers', [2, 4, 6],
    config_fast, train_dataset, val_dataset,
    train_model, compute_bleu,
)
save_results(results_E8, 'E8_layers')

for n, bleu in zip(results_E8['values'], results_E8['bleu']):
    print(f"  n_layers={n}  BLEU={bleu:.2f}")

Translation sweep: n_layers in [2, 4, 6]

--- n_layers = 2 ---
Starting training for 10 epochs, using cuda.

Epoch 1
Train -   loss: 3.79     top-1: 0.37    top-5: 0.53    top-10: 0.61
Eval -    loss: 3.62     top-1: 0.38    top-5: 0.55    top-10: 0.62
We lost no time in sending him to the hospital.
Nous n'êtes pas d'argent.

Epoch 2
Train -   loss: 3.24     top-1: 0.43    top-5: 0.60    top-10: 0.66
Eval -    loss: 3.10     top-1: 0.43    top-5: 0.61    top-10: 0.67
Tom heard a noise and went outside to see what it was.
Tom a dit qu'il n'a rien à Tom.

Epoch 3
Train -   loss: 2.99     top-1: 0.45    top-5: 0.63    top-10: 0.70
Eval -    loss: 2.85     top-1: 0.46    top-5: 0.64    top-10: 0.71
Take two of these red pills.
Fais à la maison.

Epoch 4
Train -   loss: 2.82     top-1: 0.48    top-5: 0.65    top-10: 0.72
Eval -    loss: 2.68     top-1: 0.49    top-5: 0.67    top-10: 0.73
I know you're scared.
Je connais vous.

Epoch 5
Train -   loss: 2.65     top-1: 0.49    top-5: 0.68    t

In [73]:
# E9 : Sweep du facteur d'expansion E
# Using valeurs : [1, 2, 4]

results_E9 = run_translation_sweep(
    'expand', [1, 2, 4],
    config_fast, train_dataset, val_dataset,
    train_model, compute_bleu,
)
save_results(results_E9, 'E9_expand')

for e, bleu in zip(results_E9['values'], results_E9['bleu']):
    print(f"  expand={e}  BLEU={bleu:.2f}")

Translation sweep: expand in [1, 2, 4]

--- expand = 1 ---
Starting training for 10 epochs, using cuda.

Epoch 1
Train -   loss: 3.98     top-1: 0.34    top-5: 0.50    top-10: 0.58
Eval -    loss: 3.79     top-1: 0.35    top-5: 0.52    top-10: 0.59
I'm sure that you'll succeed.
Je sais.

Epoch 2
Train -   loss: 3.36     top-1: 0.40    top-5: 0.58    top-10: 0.65
Eval -    loss: 3.23     top-1: 0.41    top-5: 0.59    top-10: 0.66
You're still green.
Vous pouvez.

Epoch 3
Train -   loss: 3.11     top-1: 0.43    top-5: 0.61    top-10: 0.68
Eval -    loss: 2.98     top-1: 0.44    top-5: 0.62    top-10: 0.69
That rings a bell.
C'est une bonne idée.

Epoch 4
Train -   loss: 2.92     top-1: 0.46    top-5: 0.64    top-10: 0.70
Eval -    loss: 2.81     top-1: 0.47    top-5: 0.65    top-10: 0.71
Is this seat empty?
Cela ?

Epoch 5
Train -   loss: 2.77     top-1: 0.47    top-5: 0.66    top-10: 0.72
Eval -    loss: 2.67     top-1: 0.48    top-5: 0.67    top-10: 0.73
Tom's French is much better tha

In [74]:
# E10 : Sweep du dropout
# Using valeurs : [0.0, 0.1, 0.3]

results_E10 = run_translation_sweep(
    'dropout', [0.0, 0.1, 0.3],
    config_fast, train_dataset, val_dataset,
    train_model, compute_bleu,
)
save_results(results_E10, 'E10_dropout')

for d, bleu in zip(results_E10['values'], results_E10['bleu']):
    print(f"  dropout={d}  BLEU={bleu:.2f}")

Translation sweep: dropout in [0.0, 0.1, 0.3]

--- dropout = 0.0 ---
Starting training for 10 epochs, using cuda.

Epoch 1
Train -   loss: 3.64     top-1: 0.38    top-5: 0.55    top-10: 0.62
Eval -    loss: 3.50     top-1: 0.39    top-5: 0.56    top-10: 0.63
I know you're busy, too.


Epoch 2
Train -   loss: 3.04     top-1: 0.45    top-5: 0.63    top-10: 0.69
Eval -    loss: 2.99     top-1: 0.45    top-5: 0.63    top-10: 0.69
They won't start the meeting without us.


Epoch 3
Train -   loss: 2.75     top-1: 0.48    top-5: 0.67    top-10: 0.73
Eval -    loss: 2.72     top-1: 0.48    top-5: 0.67    top-10: 0.73
The lightbulb has just burned out.
Le.

Epoch 4
Train -   loss: 2.53     top-1: 0.51    top-5: 0.70    top-10: 0.76
Eval -    loss: 2.53     top-1: 0.51    top-5: 0.70    top-10: 0.75
My wife used to stay home, but she works now.
Mon travail, s'il vous plaît.

Epoch 5
Train -   loss: 2.35     top-1: 0.54    top-5: 0.73    top-10: 0.78
Eval -    loss: 2.37     top-1: 0.53    top-5:

## Tableau récapitulatif des résultats

In [75]:
# Tableau récapitulatif final de toutes les expériences Mamba
import pandas as pd

summary_data = []

# E1 : Traduction
summary_data.append({
    'Expérience': 'E1 - Traduction EN→FR',
    'Métrique clé': 'BLEU',
    'Valeur': f"{results_E1.get('bleu', float('nan')):.2f}",
    'Paramètres': f"{results_E1['total_params']:,}",
})

# E3 : Selective Copying
summary_data.append({
    'Expérience': 'E3 - Selective Copying',
    'Métrique clé': 'Val accuracy',
    'Valeur': f"{results_E3['final_val_acc']:.4f}",
    'Paramètres': '-',
})

# E4 : Induction Heads
summary_data.append({
    'Expérience': 'E4 - Induction Heads (train)',
    'Métrique clé': 'Val accuracy',
    'Valeur': f"{results_E4['final_val_acc']:.4f}",
    'Paramètres': f"train_len={results_E4['train_len']}",
})

# E4 : Extrapolation à L=2048
if 2048 in results_E4['extrapolation']:
    summary_data.append({
        'Expérience': 'E4 - Induction Heads (L=2048)',
        'Métrique clé': 'Val accuracy',
        'Valeur': f"{results_E4['extrapolation'][2048]:.4f}",
        'Paramètres': '-',
    })

# E5 : Full Mamba vs non-sélective
full_acc = [c['final_val_acc'] for c in results_E5['configs'] if 'Full' in c['name']][0]
none_acc = [c['final_val_acc'] for c in results_E5['configs'] if 'non-selective' in c['name']][0]
summary_data.append({
    'Expérience': 'E5 - Full Mamba',
    'Métrique clé': 'Val accuracy',
    'Valeur': f"{full_acc:.4f}",
    'Paramètres': f"vs {none_acc:.4f} sans sélection",
})

# E6 : d_state optimal
best_N_idx = max(range(len(results_E6['val_acc'])), key=lambda i: results_E6['val_acc'][i])
summary_data.append({
    'Expérience': 'E6 - Best d_state',
    'Métrique clé': 'Val accuracy',
    'Valeur': f"{results_E6['val_acc'][best_N_idx]:.4f}",
    'Paramètres': f"N={results_E6['d_state_values'][best_N_idx]}",
})

# E7 : A init best
best_init_idx = max(range(len(results_E7['inits'])), key=lambda i: results_E7['inits'][i]['final_val_acc'])
summary_data.append({
    'Expérience': 'E7 - Best A init',
    'Métrique clé': 'Val accuracy',
    'Valeur': f"{results_E7['inits'][best_init_idx]['final_val_acc']:.4f}",
    'Paramètres': results_E7['inits'][best_init_idx]['A_init'],
})

# E11 : Scaling
L_min = results_E11['mamba']['seq_len'][0]
L_max = results_E11['mamba']['seq_len'][-1]
ratio = results_E11['mamba']['fwd_ms'][-1] / results_E11['mamba']['fwd_ms'][0]
summary_data.append({
    'Expérience': 'E11 - Mamba scaling',
    'Métrique clé': f'Time ratio L={L_max}/L={L_min}',
    'Valeur': f"{ratio:.1f}x",
    'Paramètres': f"for L x {L_max//L_min}",
})


# E8 : best n_layers
best_E8 = max(range(len(results_E8['bleu'])), key=lambda i: results_E8['bleu'][i])
summary_data.append({
    'Expérience': 'E8 - Best n_layers',
    'Métrique clé': 'BLEU',
    'Valeur': f"{results_E8['bleu'][best_E8]:.2f}",
    'Paramètres': f"n_layers={results_E8['values'][best_E8]}",
})

# E9 : best expand
best_E9 = max(range(len(results_E9['bleu'])), key=lambda i: results_E9['bleu'][i])
summary_data.append({
    'Expérience': 'E9 - Best expand',
    'Métrique clé': 'BLEU',
    'Valeur': f"{results_E9['bleu'][best_E9]:.2f}",
    'Paramètres': f"expand={results_E9['values'][best_E9]}",
})

# E10 : best dropout
best_E10 = max(range(len(results_E10['bleu'])), key=lambda i: results_E10['bleu'][i])
summary_data.append({
    'Expérience': 'E10 - Best dropout',
    'Métrique clé': 'BLEU',
    'Valeur': f"{results_E10['bleu'][best_E10]:.2f}",
    'Paramètres': f"dropout={results_E10['values'][best_E10]}",
})


df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

# Sauvegarder en CSV pour le rapport
df_summary.to_csv('results/summary_table.csv', index=False)
print("\nSaved: results/summary_table.csv")

                   Expérience            Métrique clé Valeur               Paramètres
        E1 - Traduction EN→FR                    BLEU  36.19                9,201,166
       E3 - Selective Copying            Val accuracy 0.9670                        -
 E4 - Induction Heads (train)            Val accuracy 1.0000            train_len=128
E4 - Induction Heads (L=2048)            Val accuracy 0.8600                        -
              E5 - Full Mamba            Val accuracy 0.9670 vs 0.2535 sans sélection
            E6 - Best d_state            Val accuracy 0.9670                     N=16
             E7 - Best A init            Val accuracy 0.9780                   random
          E11 - Mamba scaling Time ratio L=8192/L=128  58.3x               for L x 64
           E8 - Best n_layers                    BLEU  18.42               n_layers=6
             E9 - Best expand                    BLEU  25.90                 expand=4
           E10 - Best dropout                    BLEU 